# To do list:

* Input: Start date, End date, short/medium/long
* From Start date, End date => List all range


In [3]:
import os, json
import pandas as pd
import numpy as np
import mysql.connector
from dotenv import load_dotenv
from typing import List, Union, Optional

load_dotenv()

# ----------------------------------------------------------------------
# DB Connection
# ----------------------------------------------------------------------
def get_connection():
    """Efficient MySQL connection"""
    return mysql.connector.connect(
        host=os.getenv("MYSQL_HOST"),
        user=os.getenv("MYSQL_USER"),
        password=os.getenv("MYSQL_PASSWORD"),
        database=os.getenv("MYSQL_DATABASE"),
        connection_timeout=5
    )

# ----------------------------------------------------------------------
# Main Function
# ----------------------------------------------------------------------
def get_latest_price(
    coin_symbol: Union[List[str], str],
    time_horizons: Union[List[str], str],
    start_date: Optional[str] = None,
    end_date: Optional[str] = None
) -> str:
    """
    Keep full kline data but compress funding rates with sparse timestamps and summary stats.
    Open interest remains 4h-aggregated for alignment.
    """
    if isinstance(coin_symbol, str):
        coin_symbol = [coin_symbol]
    if isinstance(time_horizons, str):
        time_horizons = [time_horizons]

    conn = get_connection()
    cursor = conn.cursor(dictionary=True)
    results = {}

    def run_query(query: str, params: tuple = ()):
        cursor.execute(query, params)
        rows = cursor.fetchall()
        return pd.DataFrame(rows) if rows else pd.DataFrame()

    # --- Date filter ---
    date_filter = ""
    date_params = []
    if start_date:
        date_filter += " AND DATE(open_time) >= %s"
        date_params.append(start_date)
    if end_date:
        date_filter += " AND DATE(open_time) <= %s"
        date_params.append(end_date)

    # ------------------------------------------------------------------
    for sym in coin_symbol:
        sym = sym.lower().strip()
        sym_results = {}

        for horizon in time_horizons:
            horizon = horizon.lower().strip()
            dfs = {}

            if horizon == "short":
                # --- 4h kline (keep all OHLCV) ---
                dfs["kline"] = run_query(
                    f"""
                    SELECT close_time, open_price, high_price, low_price, close_price, volume
                    FROM finance_services.crypto_kline_hours
                    WHERE `interval`='4h' AND symbol=%s {date_filter}
                    ORDER BY close_time
                    """,
                    (sym, *date_params)
                )

                # --- Funding rates (hybrid compression) ---
                df_fund = run_query(
                    f"""
                    SELECT funding_time, funding_rate
                    FROM finance_services.futures_funding_rates
                    WHERE symbol=%s
                    {" AND DATE(funding_time) BETWEEN %s AND %s" if start_date and end_date else ""}
                    ORDER BY funding_time
                    """,
                    (sym, *(date_params if len(date_params) == 2 else []))
                )

                if not df_fund.empty:
                    df_fund = df_fund.sort_values("funding_time").reset_index(drop=True)
                    df_fund["funding_delta"] = df_fund["funding_rate"].diff().fillna(0) * 1e4
                    base_rate = float(df_fund["funding_rate"].iloc[0])
                    deltas = df_fund["funding_delta"].iloc[1:]
                    dfs["funding"] = {
                        "base_rate": base_rate,
                        "stats": {
                            "avg_delta": round(float(deltas.mean()), 3),
                            "max_delta": round(float(deltas.max()), 3),
                            "min_delta": round(float(deltas.min()), 3),
                            "volatility": round(float(deltas.std()), 3)
                        },
                        "series": df_fund.iloc[::max(1, len(df_fund)//8)][["funding_time","funding_rate"]]
                                  .rename(columns={"funding_time": "ts", "funding_rate": "rate"})
                                  .round(8)
                                  .to_dict(orient="records"),
                        "count": len(df_fund)
                    }
                else:
                    dfs["funding"] = {}

                # --- Open Interest (4h aggregation) ---
                dfs["open_interest"] = run_query(
                    f"""
                    SELECT 
                        DATE_FORMAT(MIN(timestamp),'%Y-%m-%d %H:00:00') AS ts,
                        AVG(open_interest_usd) AS open_interest_usd,
                        AVG(open_interest_coin) AS open_interest_coin
                    FROM finance_services.futures_open_interests
                    WHERE symbol=%s
                    {" AND DATE(timestamp) BETWEEN %s AND %s" if start_date and end_date else ""}
                    GROUP BY FLOOR(UNIX_TIMESTAMP(timestamp)/(4*3600))
                    ORDER BY ts
                    """,
                    (sym, *(date_params if len(date_params) == 2 else []))
                )

            elif horizon == "short-simulator":
                dfs["kline"] = run_query(
                    f"""
                    SELECT close_time, close_price as price from finance_services.crypto_kline_hours
                    WHERE `interval`='1h' AND symbol=%s {date_filter}
                    ORDER BY close_time
                    """,
                    (sym, *date_params)
                )
                      

            else: 
                continue

            # --- Clean and summarize ---
            summary = []
            for name, df in dfs.items():
                if isinstance(df, pd.DataFrame):
                    if "volume" in df.columns:
                        df = df[df["volume"] > 0].copy()
                    float_cols = df.select_dtypes(include="float").columns
                    df.loc[:, float_cols] = df[float_cols].round(3)
                    time_col = next((c for c in ["open_time", "close_time", "funding_time", "timestamp", "ts"] if c in df.columns), None)
                    tmin = df[time_col].min() if time_col else None
                    tmax = df[time_col].max() if time_col else None
                    summary.append({
                        "dataset": name,
                        "rows": len(df),
                        "columns": list(df.columns),
                        "start_time": tmin,
                        "end_time": tmax
                    })
                    dfs[name] = df
                else:
                    summary.append({
                        "dataset": name,
                        "rows": len(df.get("series", [])),
                        "columns": list(df.keys()),
                        "start_time": df["series"][0]["ts"] if df.get("series") else None,
                        "end_time": df["series"][-1]["ts"] if df.get("series") else None
                    })

            sym_results[horizon] = {
                "data": {k: (v.to_dict(orient="records") if isinstance(v, pd.DataFrame) else v) for k, v in dfs.items()},
                "summary": summary
            }

        results[sym] = sym_results

    cursor.close()
    conn.close()
    return json.dumps(results, default=str, separators=(",", ":"))


In [9]:
get_latest_price('eth', 'short-simulator', '2025-10-09', '2025-10-12' )

'{"eth":{"short-simulator":{"data":{"kline":[{"close_time":"2025-10-09 01:00:00","price":4538.46},{"close_time":"2025-10-09 02:00:00","price":4523.48},{"close_time":"2025-10-09 03:00:00","price":4515.27},{"close_time":"2025-10-09 04:00:00","price":4503.66},{"close_time":"2025-10-09 05:00:00","price":4525.59},{"close_time":"2025-10-09 06:00:00","price":4521.32},{"close_time":"2025-10-09 07:00:00","price":4525.72},{"close_time":"2025-10-09 08:00:00","price":4511.29},{"close_time":"2025-10-09 09:00:00","price":4478.99},{"close_time":"2025-10-09 10:00:00","price":4412.82},{"close_time":"2025-10-09 11:00:00","price":4450.8},{"close_time":"2025-10-09 12:00:00","price":4455.04},{"close_time":"2025-10-09 13:00:00","price":4454.69},{"close_time":"2025-10-09 14:00:00","price":4429.55},{"close_time":"2025-10-09 15:00:00","price":4426.6},{"close_time":"2025-10-09 16:00:00","price":4328.0},{"close_time":"2025-10-09 17:00:00","price":4334.82},{"close_time":"2025-10-09 18:00:00","price":4354.7},{"clo

In [17]:
from datetime import datetime, timedelta

class DateRangeSplitter:
    def __init__(self, start_date: str, end_date: str, chunk_days: int):
        self.start_date = datetime.strptime(start_date, "%Y-%m-%d")
        self.end_date = datetime.strptime(end_date, "%Y-%m-%d")
        self.chunk_days = chunk_days
        self._ranges = self._generate_ranges()

    def _generate_ranges(self):
        """Internal: generate list of date range dicts."""
        ranges = []
        current = self.start_date
        while current < self.end_date:
            next_end = current + timedelta(days=self.chunk_days)
            if next_end > self.end_date:
                next_end = self.end_date
            ranges.append({
                "start": current.strftime("%Y-%m-%d"),
                "end": next_end.strftime("%Y-%m-%d"),
            })
            current = next_end
        return ranges

    # --- Public methods ---
    def get_meta(self):
        """Return metadata about the split."""
        return {
            "start_date": self.start_date.strftime("%Y-%m-%d"),
            "end_date": self.end_date.strftime("%Y-%m-%d"),
            "chunk_days": self.chunk_days,
            "total_days": (self.end_date - self.start_date).days,
            "chunk_count": len(self._ranges)
        }

    def get_ranges(self):
        """Return the full detailed list of ranges."""
        return self._ranges

    def get_array(self):
        """Return only [start, end] pairs."""
        return [[r["start"], r["end"]] for r in self._ranges]

    def __repr__(self):
        return f"<DateRangeSplitter chunks={len(self._ranges)} days_per_chunk={self.chunk_days}>"


In [18]:
splitter = DateRangeSplitter("2025-06-01", "2025-11-03", 3)

print("Meta:")
print(splitter.get_meta())

print("\nDetailed ranges:")
for r in splitter.get_ranges():
    print(r)

print("\nArray form:")
print(splitter.get_array())


Meta:
{'start_date': '2025-06-01', 'end_date': '2025-11-03', 'chunk_days': 3, 'total_days': 155, 'chunk_count': 52}

Detailed ranges:
{'start': '2025-06-01', 'end': '2025-06-04'}
{'start': '2025-06-04', 'end': '2025-06-07'}
{'start': '2025-06-07', 'end': '2025-06-10'}
{'start': '2025-06-10', 'end': '2025-06-13'}
{'start': '2025-06-13', 'end': '2025-06-16'}
{'start': '2025-06-16', 'end': '2025-06-19'}
{'start': '2025-06-19', 'end': '2025-06-22'}
{'start': '2025-06-22', 'end': '2025-06-25'}
{'start': '2025-06-25', 'end': '2025-06-28'}
{'start': '2025-06-28', 'end': '2025-07-01'}
{'start': '2025-07-01', 'end': '2025-07-04'}
{'start': '2025-07-04', 'end': '2025-07-07'}
{'start': '2025-07-07', 'end': '2025-07-10'}
{'start': '2025-07-10', 'end': '2025-07-13'}
{'start': '2025-07-13', 'end': '2025-07-16'}
{'start': '2025-07-16', 'end': '2025-07-19'}
{'start': '2025-07-19', 'end': '2025-07-22'}
{'start': '2025-07-22', 'end': '2025-07-25'}
{'start': '2025-07-25', 'end': '2025-07-28'}
{'start': '

In [ ]:
# Prompt config
technical_analysis_prompt = """
Bạn là chuyên gia phân tích kỹ thuật crypto chuyên sâu, sử dụng logic phân tích đa khung thời gian 4H kết hợp dữ liệu thực tế gồm:
- Khung giờ mà bạn sẽ được cung cấp cũng như sử dụng là giờ Việt Nam, asia, UTC+7
- Giá hiện tại, giá mở cửa, biên độ dao động 24h.
- Các mức hỗ trợ, kháng cự.
- Volume, OI, Funding rate.
- Biểu đồ giá hoặc dữ liệu nến theo khung 4 giờ
## 1. Diễn biến giá (4H gần nhất)
* Giá hiện tại, giá mở cửa, biên độ 24h (% thay đổi).
* Mô tả hành động giá 4H gần nhất (mẫu nến, hướng xu hướng, độ dốc EMA, tín hiệu RSI/MACD nếu có).
* So sánh **Volume hiện tại với trung bình 20 kỳ** (VD: “Volume tăng 150% so với trung bình, xác nhận đà mua chủ động”).
* **OI**: Tỷ lệ thay đổi trong 24h (%), phân tích tương quan với giá.

  * OI ↑ & giá ↑ → dòng tiền mới tham gia (tích cực)
  * OI ↑ & giá ↓/đi ngang → trap/ép long-short
* **Funding Rate**: Xu hướng (↑ / ↓ / trung lập), cảnh báo nếu lệch cân (VD: “Funding 0.08%, cao → rủi ro short squeeze”).
## 2. Hỗ trợ – Kháng cự ngắn hạn
Liệt kê 2 mức hỗ trợ & 2 mức kháng cự rõ ràng (ghi giá cụ thể hoặc vùng biên ±0.5%).
→ Gợi ý ghi chú thêm “Hỗ trợ 1: ... | Kháng cự 1: ...”
## 3. Xác suất diễn biến (24–72 giờ tới)

| Kịch bản            | Xác suất (%) | Giải thích                                                                      |
| ------------------- | ------------ | ------------------------------------------------------------------------------- |
| Tăng / hồi kỹ thuật | XX%          | Giải thích xác suất hiện tại dựa vào những yếu tố định tính và định lượng nào? |
| Giảm / thủng hỗ trợ | YY%          | Giải thích xác suất hiện tại dựa vào những yếu tố định tính và định lượng nào? |

## 4. Gợi ý điểm vào – ra (Quản trị rủi ro định lượng)
**Long Setup:**
* Entry: vùng giá …
* Stop Loss: dựa theo **1.5x–2x ATR(14)** dưới hỗ trợ gần nhất
* Take Profit:

  * TP1: kháng cự gần nhất (chốt 50%)
  * TP2: kháng cự mạnh (chốt 50%)
* Khi TP1 đạt → **dời SL về điểm hòa vốn (Break-even)**
**Short Setup:**
* Entry: vùng giá …
* Stop Loss: **1.5x–2x ATR(14)** trên kháng cự gần nhất
* Take Profit tương tự (TP1/TP2 + quản trị dời SL)
## 5. Tổng hợp chiến lược

| Khung | Hỗ trợ | Kháng cự | Xu hướng | Kịch bản chính | Kịch bản phụ |
| ----- | ------ | -------- | -------- | -------------- | ------------ |
## 6. Gợi ý hành động (24–72h)

* **Trader ngắn hạn:** tập trung theo dõi vùng ... với điều kiện volume xác nhận hoặc OI tăng.
* Nếu Funding Rate cao → cảnh giác **trap tăng**; nếu thấp bất thường → đề phòng **short squeeze**.
* Chỉ vào lệnh khi có **xác nhận volume ≥120% trung bình 20 kỳ**.
"""

reviewer_prompt = f"""
You are a senior crypto-technical-analysis reviewer.  
You will be given two pieces of information:
  
1. The agent’s response to that prompt, which made predictions/analyses.
2. The future 

Your task: **Evaluate** the agent’s response *in the context of back-testing* (i.e., you know the outcome since the events have already happened). Use the following criteria:

A. Correctness & factual alignment  
   - Did the agent correctly identify key support/resistance levels based on the data provided?  
   - Did the outcome match the agent’s predictions (price move, breakout, reversal) or was the prediction off-mark?  
   - Did the agent use actual data (volume, OI, funding rate) in a meaningful way?

B. Analytical quality  
   - Is the multi-timeframe logic (4H → 1D → 1W) applied appropriately?  
   - Are trade suggestions (entry, TP, SL) coherent and justified given the data?  
   - Are probabilities given (e.g., “X% chance of increase”) reasonable and explained?

C. Style & completeness  
   - Is the tone professional, neutral, and consistent with a tactical analyst?  
   - Are all required sections present: 4H summary, 2 supports & resistances, probabilities, trade suggestions, table of strategies, action summary, and final “Gợi ý thêm…” line?  
   - Are there hallucinations or unsupported claims?
   - Final output should be in Vietnamese
D. Back-testing relevance  
   - Since this is back-testing, highlight where the agent’s predictions succeeded/failed.  
   - Suggest improvements: e.g., “If agent had included post-event funding rate change of X%, the prediction would be stronger/should be adjusted.”

E. Score & commentary  
   - Provide a numerical score from 0-10 for the overall response.  
   - Provide short commentary (2-3 paragraphs) explaining the score and key strengths/weaknesses.  
   - Finally, give one concrete improvement suggestion for next run.

**Important**: Use the tools available (like a code tool for calculations, a web-search tool for checking historical data/funding/OI) if needed.  
Output format:  

"""

trader_prompt = """
Bạn là chuyên gia mô phỏng giao dịch crypto (Backtesting Analyst).
Mục tiêu: đánh giá độ chính xác của bản phân tích kỹ thuật bằng cách giả lập các lệnh theo đúng chiến lược kỹ thuật, sau đó so sánh với dữ liệu thực tế để tính hiệu suất và sai lệch.
 Bạn sẽ được cung cấp:
Bản Technical Analysis (phân tích kỹ thuật):
Đây là cơ sở duy nhất để bạn ra quyết định vào/ra lệnh.
Bạn phải tuân thủ tuyệt đối, không được dựa vào dữ liệu thực tế khi đặt lệnh.

Dữ liệu thực tế (price history, volume, funding, OI...): Dùng chỉ để kiểm chứng kết quả, tính toán thời điểm chạm TP/SL, lãi/lỗ, và độ chính xác của phân tích kỹ thuật.
Quy tắc mô phỏng lệnh
Điểm vào/ra rõ ràng và có thời gian cụ thể (UTC+7)
Ví dụ:

Ngày 10/10/2025 – 10:00:00 vào lệnh Long ở 10000 USDT,
Chốt lời tại 11000 USDT (ngày 10/10/2025 – 18:00:00).

Số lượng lệnh
Có thể mở nhiều lệnh nếu bản phân tích kỹ thuật cho phép (ví dụ nhiều vùng hỗ trợ/kháng cự).
Mỗi lệnh phải được ghi lại riêng biệt với lý do kỹ thuật (RSI, MA, Breakout…).
Đánh giá định tính

Độ chính xác: …

Điều kiện phân tích kỹ thuật hoạt động tốt: (ví dụ: khi volume cao / thị trường có trend).

Sai lệch hoặc nguyên nhân thất bại: (funding lệch, volume yếu, biến động bất thường).

Kết luận: phân tích kỹ thuật đúng / sai / một phần đúng, kèm đề xuất cải thiện (thêm xác nhận RSI, ATR filter, khung 1D,…).

Nguyên tắc bắt buộc cho Agent

Không được “ăn gian” bằng cách nhìn dữ liệu thực để chọn điểm vào tốt hơn.

Không được sửa lệnh sau khi biết kết quả.

Mục tiêu không phải tối đa hóa lợi nhuận, mà là đánh giá độ tin cậy của phân tích kỹ thuật.
"""

In [ ]:

from agents import Agent, Runner 
async def run_trading_agent():
    agent_trade = Agent(
        name="trading-agent",
        model="gpt-5-mini",
        instructions=trader_prompt,
    )
    
    technical_analysis = """
Dưới đây là phân tích kỹ thuật ETH trên khung 4H (giờ VN UTC+7) dựa trên dữ liệu bạn cung cấp (kline 4H đến 2025-10-09 03:00). Mình trình bày theo cấu trúc yêu cầu.

Tổng quan nhanh

Giá hiện tại (close 4H gần nhất): 4,515.27 USD (close_time 2025-10-09 03:00)
Giá mở cửa cây 4H hiện tại: 4,447.15 USD
Biên độ 24h: +0.80% (so với close 2025-10-08 03:00 = 4,479.44 → hiện tại 4,515.27)
Lưu ý: phân tích dựa trên dữ liệu 4H, chỉ ra các mức theo vùng ±0.5% khi cần.
Diễn biến giá (4H gần nhất)
Hành động giá 4H gần nhất:
Cây 4H cuối: open 4,447.15 → high 4,558.00 → low 4,439.00 → close 4,515.27. Đây là cây tăng mạnh với thân lớn và high phá vòm kháng cự ngắn hạn (breakout intraday), đóng cửa ở phần trên thân → cấu trúc ngắn hạn chuyển sang tích cực.
Hành động 4H trước đó là dao động side-down, sau đó có một cú tăng mạnh (volume lớn) tạo đà phục hồi.
EMA & momentum (tóm tắt suy luận từ hành động giá):
EMA ngắn hạn (EMA20) có độ dốc nhẹ hướng lên; EMA trung/ dài (EMA50) đang dao động ngang → cấu trúc chuyển từ side sang tăng ngắn hạn (momentum mới xuất hiện nhưng chưa mạnh liên tục).
MACD/RSI (không có số liệu trực tiếp): suy luận — MACD histogram có khả năng dương (đà tăng vừa xuất hiện); RSI 4H ước tính ở vùng trung tính ~55–65 (chưa quá mua).
Kết luận: xu hướng 4H đang nghiêng TĂNG ngắn hạn nhưng độ xác nhận còn phụ thuộc volume & OI.
Volume: so sánh với trung bình 20 kỳ (4H):
Trung bình 20 kỳ (volume) ≈ 79,946 (ước tính từ dữ liệu 20 cây 4H gần nhất).
Volume hiện tại = 94,654 → ~118% so với trung bình 20 kỳ (Volume tăng ~18% so với MA20) → xác nhận đà mua nhưng chưa đạt ngưỡng mạnh (≥120% theo rule bạn nêu).
Open Interest (OI):
OI (USD) gần nhất có trong dữ liệu: 2025-10-08 23:00 = 8,625,865,164 USD. So sánh với 24h trước (2025-10-08 03:00 = 8,821,583,172 USD) → OI giảm khoảng -2.2% trong ~24h.
Phân tích tương quan giá/OI: Giá ↑ (+0.8%) trong khi OI ↓ (-2.2%) → dấu hiệu chủ yếu là short-covering (cover vị thế short) hơn là dòng tiền mới mở long mạnh. Có thể là rally do cover/profit taking, chưa có dòng tiền mới lớn đẩy OI tăng.
Funding rate:
Dãy funding gần đây quanh 1e-5 → 1e-4, base_rate hiện rất sát 0 (−1.572e-05). Xu hướng funding: gần như trung lập → không có áp lực ưu thế long/short mạnh. Lưu ý có vài phiên funding dương nhỏ (tức longs trả shorts) nhưng mức rất thấp → không báo động short/long squeeze.
2. Hỗ trợ – Kháng cự ngắn hạn (vùng ±0.5%)

Hỗ trợ 1: 4,445 – 4,455 USD (vùng cluster đáy/đóng cửa 10-08 → 10-09)
Hỗ trợ 2: 4,360 – 4,380 USD (vùng đáy và vùng breakout ngày 10-02 → 10-03)
Kháng cự 1: 4,550 – 4,565 USD (đỉnh cây 4H gần nhất 4,558)
Kháng cự 2: 4,690 – 4,730 USD (vùng tâm lý + đỉnh vùng 4,673–4,722 và mức 4,693 trước đó) Ghi chú: Hỗ trợ 1: 4,445–4,455 | Kháng cự 1: 4,550–4,565
3. Xác suất diễn biến (24–72 giờ tới)

Kịch bản	Xác suất (%)	Giải thích
Tăng / hồi kỹ thuật	55%	Cây 4H gần nhất tăng mạnh, volume tăng ~118% so với MA20 → có lực mua ngắn hạn. Tuy nhiên OI giảm (−2.2%) cho thấy phần lớn là short-covering, chưa có dòng tiền mới mạnh → khả năng tiếp tục hồi có nhưng hạn chế.
Giảm / thủng hỗ trợ	45%	OI giảm và funding trung lập → thiếu xác nhận dòng tiền mới; nếu vùng 4,445 bị từ chối/khối lượng giảm thì có thể xuất hiện bán tháo/test hỗ trợ 2 (~4,360). Rủi ro ngược chiều vẫn khá lớn.
(Ý nghĩa: phân chia 55/45 do tín hiệu giá + volume hơi thiên tăng nhưng OI không hỗ trợ.)

4. Gợi ý điểm vào – ra (Quản trị rủi ro định lượng)
Lưu ý: không có ATR(14) trực tiếp trong dữ liệu; mình ước lượng ATR(14) 4H ≈ 110 USD (dựa trên biên độ các nến gần đây). Bạn thay giá trị ATR(14) thực tế nếu có.

Long Setup:

Entry (chiến lược an toàn): chờ pullback về vùng hỗ trợ 1 = 4,445–4,455 USD; hoặc momentum-entry khi giá xuyên và đóng ổn định trên 4,565 USD (breakout confirmed).
Stop Loss: đặt theo 1.5x–2x ATR(14) dưới hỗ trợ gần nhất.
Ví dụ với ATR≈110: SL = 1.5×110 = 165 USD → nếu entry tại 4,450 (mid hỗ trợ) → SL ~4,285 USD.
Nếu entry theo breakout trên 4,565 → SL = 1.5×ATR dưới breakout ≈ 4,400 USD (tùy điểm vào).
Take Profit:
TP1: 4,550–4,565 (kháng cự gần nhất) — chốt 50%
TP2: 4,690–4,730 (kháng cự mạnh) — chốt 50%
Quản trị: khi TP1 đạt → dời SL về breakeven + chút room (phí/spread).
Short Setup:

Entry: ở vùng kháng cự 1 nếu bị từ chối 4,550–4,565 hoặc trên tín hiệu thất bại breakout (bear trap); hoặc vào short khi thủng hỗ trợ 1 và đóng dưới nó.
Stop Loss: 1.5x–2x ATR(14) trên kháng cự gần nhất.
Ví dụ ATR≈110 → SL ~+165 USD trên vùng kháng cự (nếu entry 4,560 → SL ~4,725).
Take Profit:
TP1: 4,445–4,455 (hỗ trợ gần) — chốt 50%
TP2: 4,360–4,380 (hỗ trợ sâu) — chốt 50%
Quản trị: nếu TP1 đạt → dời SL về hòa vốn hoặc trailing SL.
5. Tổng hợp chiến lược (tóm tắt)

Khung	Hỗ trợ	Kháng cự	Xu hướng	Kịch bản chính	Kịch bản phụ
4H	4,445 / 4,370	4,560 / 4,700	Ngắn hạn: nghiêng tăng	Hồi kỹ thuật tiếp tục lên 4,560 → test 4,700	Bị từ chối tại 4,560 → test lại 4,445 → có thể thủng xuống 4,370
6. Gợi ý hành động (24–72h)

Trader ngắn hạn: theo dõi vùng 4,445–4,565. Vào long chỉ khi:
(A) Pullback về hỗ trợ 4,445–4,455 với volume ≥120% MA20 hoặc
(B) Break & close ổn định trên 4,565 với volume tăng và OI bắt đầu tăng (xác nhận dòng tiền mới).
Nếu Funding Rate tăng đột biến dương → cảnh giác trap tăng (long crowded). Nếu Funding âm đột biến → cảnh giác short squeeze.
Quy tắc entry: chỉ vào lệnh khi volume xác nhận ≥120% trung bình 20 kỳ (hiện volume ~118% → gần đạt, vẫn còn thiếu chút).
Giữ quản trị rủi ro: position size tính để rủi ro trên mỗi trade ≤1–2% vốn tùy profile.
Kết luận ngắn gọn

ETH trên 4H vừa có cây tăng mạnh và volume cao (khoảng +18% so với MA20) → tín hiệu hồi tăng ngắn hạn. Tuy nhiên OI giảm (-2.2% trong 24h) cho thấy động lực nhiều là short-covering chứ chưa phải dòng tiền long mới mạnh. Funding gần trung lập → không có squeeze rõ rệt.
Chiến lược khôn ngoan: ưu tiên chờ pullback có volume/ OI xác nhận hoặc chờ breakout đóng nến 4H trên 4,565 với OI tăng trước khi vào long lớn. Nếu thấy từ chối tại vùng kháng cự, sẵn sàng chơi short với quản trị rủi ro chặt.
    """
    real_price_data = """
    '{"eth":{"short-simulator":{"data":{"kline":[{"close_time":"2025-10-09 01:00:00","price":4538.46},{"close_time":"2025-10-09 02:00:00","price":4523.48},{"close_time":"2025-10-09 03:00:00","price":4515.27},{"close_time":"2025-10-09 04:00:00","price":4503.66},{"close_time":"2025-10-09 05:00:00","price":4525.59},{"close_time":"2025-10-09 06:00:00","price":4521.32},{"close_time":"2025-10-09 07:00:00","price":4525.72},{"close_time":"2025-10-09 08:00:00","price":4511.29},{"close_time":"2025-10-09 09:00:00","price":4478.99},{"close_time":"2025-10-09 10:00:00","price":4412.82},{"close_time":"2025-10-09 11:00:00","price":4450.8},{"close_time":"2025-10-09 12:00:00","price":4455.04},{"close_time":"2025-10-09 13:00:00","price":4454.69},{"close_time":"2025-10-09 14:00:00","price":4429.55},{"close_time":"2025-10-09 15:00:00","price":4426.6},{"close_time":"2025-10-09 16:00:00","price":4328.0},{"close_time":"2025-10-09 17:00:00","price":4334.82},{"close_time":"2025-10-09 18:00:00","price":4354.7},{"close_time":"2025-10-09 19:00:00","price":4373.05},{"close_time":"2025-10-09 20:00:00","price":4404.37},{"close_time":"2025-10-09 21:00:00","price":4350.81},{"close_time":"2025-10-09 22:00:00","price":4344.88},{"close_time":"2025-10-09 23:00:00","price":4314.79},{"close_time":"2025-10-10 00:00:00","price":4277.34},{"close_time":"2025-10-10 01:00:00","price":4334.4},{"close_time":"2025-10-10 02:00:00","price":4308.24},{"close_time":"2025-10-10 03:00:00","price":4338.0},{"close_time":"2025-10-10 04:00:00","price":4337.46},{"close_time":"2025-10-10 05:00:00","price":4357.85},{"close_time":"2025-10-10 06:00:00","price":4377.83},{"close_time":"2025-10-10 07:00:00","price":4368.09},{"close_time":"2025-10-10 08:00:00","price":4380.7},{"close_time":"2025-10-10 09:00:00","price":4373.38},{"close_time":"2025-10-10 10:00:00","price":4359.19},{"close_time":"2025-10-10 11:00:00","price":4347.13},{"close_time":"2025-10-10 12:00:00","price":4344.49},{"close_time":"2025-10-10 13:00:00","price":4368.34},{"close_time":"2025-10-10 14:00:00","price":4328.0},{"close_time":"2025-10-10 15:00:00","price":4323.39},{"close_time":"2025-10-10 16:00:00","price":4346.12},{"close_time":"2025-10-10 17:00:00","price":4322.12},{"close_time":"2025-10-10 18:00:00","price":4342.89},{"close_time":"2025-10-10 19:00:00","price":4343.02},{"close_time":"2025-10-10 20:00:00","price":4348.54},{"close_time":"2025-10-10 21:00:00","price":4356.1},{"close_time":"2025-10-10 22:00:00","price":4280.35},{"close_time":"2025-10-10 23:00:00","price":4103.11},{"close_time":"2025-10-11 00:00:00","price":4108.99},{"close_time":"2025-10-11 01:00:00","price":4093.97},{"close_time":"2025-10-11 02:00:00","price":4053.84},{"close_time":"2025-10-11 03:00:00","price":3997.56},{"close_time":"2025-10-11 04:00:00","price":3871.3},{"close_time":"2025-10-11 05:00:00","price":3949.99},{"close_time":"2025-10-11 06:00:00","price":3855.09},{"close_time":"2025-10-11 07:00:00","price":3829.72},{"close_time":"2025-10-11 08:00:00","price":3838.48},{"close_time":"2025-10-11 09:00:00","price":3733.35},{"close_time":"2025-10-11 10:00:00","price":3841.17},{"close_time":"2025-10-11 11:00:00","price":3792.26},{"close_time":"2025-10-11 12:00:00","price":3820.54},{"close_time":"2025-10-11 13:00:00","price":3789.47},{"close_time":"2025-10-11 14:00:00","price":3781.78},{"close_time":"2025-10-11 15:00:00","price":3758.81},{"close_time":"2025-10-11 16:00:00","price":3815.36},{"close_time":"2025-10-11 17:00:00","price":3829.37},{"close_time":"2025-10-11 18:00:00","price":3833.89},{"close_time":"2025-10-11 19:00:00","price":3839.99},{"close_time":"2025-10-11 20:00:00","price":3823.52},{"close_time":"2025-10-11 21:00:00","price":3832.39},{"close_time":"2025-10-11 22:00:00","price":3814.2},{"close_time":"2025-10-11 23:00:00","price":3825.13},{"close_time":"2025-10-12 00:00:00","price":3820.07},{"close_time":"2025-10-12 01:00:00","price":3825.14},{"close_time":"2025-10-12 02:00:00","price":3815.48},{"close_time":"2025-10-12 03:00:00","price":3760.61},{"close_time":"2025-10-12 04:00:00","price":3695.15},{"close_time":"2025-10-12 05:00:00","price":3735.75},{"close_time":"2025-10-12 06:00:00","price":3742.94},{"close_time":"2025-10-12 07:00:00","price":3746.79},{"close_time":"2025-10-12 08:00:00","price":3701.01},{"close_time":"2025-10-12 09:00:00","price":3719.24},{"close_time":"2025-10-12 10:00:00","price":3734.06},{"close_time":"2025-10-12 11:00:00","price":3781.61},{"close_time":"2025-10-12 12:00:00","price":3801.9},{"close_time":"2025-10-12 13:00:00","price":3806.33},{"close_time":"2025-10-12 14:00:00","price":3832.84},{"close_time":"2025-10-12 15:00:00","price":3827.99},{"close_time":"2025-10-12 16:00:00","price":3823.17},{"close_time":"2025-10-12 17:00:00","price":3843.8},{"close_time":"2025-10-12 18:00:00","price":3820.0},{"close_time":"2025-10-12 19:00:00","price":3820.36},{"close_time":"2025-10-12 20:00:00","price":3832.1},{"close_time":"2025-10-12 21:00:00","price":3810.13},{"close_time":"2025-10-12 22:00:00","price":3975.19},{"close_time":"2025-10-12 23:00:00","price":4034.59},{"close_time":"2025-10-13 00:00:00","price":4106.27}]},"summary":[{"dataset":"kline","rows":96,"columns":["close_time","price"],"start_time":"2025-10-09 01:00:00","end_time":"2025-10-13 00:00:00"}]}}}'
    """
    prompt = f"""
    Mô phỏng giao dịch BTC dựa trên phân tích kỹ thuật sau đây:
    {technical_analysis}

    Diễn biến giá thực tế
    {real_price_data}
    """

    result = await Runner.run(agent_trade, prompt)

    print(result.final_output)
    return result.final_output



In [11]:
result = await run_trading_agent()

1. Kết quả tổng hợp
| # | Thời gian vào        | Thời gian ra        | Loại lệnh | Giá vào  | Giá ra   | Kết quả (PnL%) | Thời gian giữ lệnh | Lý do                                    | Dẫn chứng từ báo cáo kỹ thuật                                      |
| - | -------------------- | ------------------- | --------- | -------- | -------- | -------------- | ------------------ | ---------------------------------------- | ------------------------------------------------------------------ |
| 1 | 09/10/2025 11:00 (UTC+7) | 10/10/2025 22:00 (UTC+7) | Long      | 4,450.80 | 4,280.35 | -3.8%          | 35h               | Pullback về hỗ trợ 4,445–4,455 (vào long) | “Hỗ trợ 1: 4,445 – 4,455 USD (vùng cluster đáy/đóng cửa 10-08 → 10-09)” |

Tổng (tổng hợp các chỉ số): Lãi/Lỗ tổng = -3.8% | Winrate = 0% (0/1) | Max drawdown = -3.8% | RR trung bình (thực tế) = -1.03 | RR trung bình (kế hoạch) ≈ 1.10

2. Tổng hợp lãi/lỗ, winrate (%), max drawdown, RR trung bình
| # | Kết quả | Lãi / Lỗ (%) | Nhận đị

In [20]:
# Run the AI workflow:
from agents import Agent, Runner, WebSearchTool, CodeInterpreterTool
from pydantic import BaseModel
from typing import Dict, List

class TechnicalAnalysis(BaseModel):
    start_date: str
    end_date: str
    analysis: str

class ReviewerAnalysis(BaseModel):
    analysis_start: str
    analysis_end: str
    review_start: str
    review_end: str
    analysis: str
    review: str
    analysis_token: int
    review_token: int


# ----------------------------------------------------------------------
# Helper: Fetch and split data for analysis + review periods
# ----------------------------------------------------------------------
def fetch_backtest_data(analysis_start: str, analysis_end: str, review_end: str, symbol: str = 'btc'):
    """
    Fetch market data for both analysis and review periods.

    Args:
        analysis_start: Start date for technical analysis period (e.g., '2025-10-01')
        analysis_end: End date for technical analysis period (e.g., '2025-10-08')
        review_end: End date for review period (e.g., '2025-10-15')
        symbol: Coin symbol (default: 'btc')

    Returns:
        dict with 'analysis_data' and 'full_data' keys
    """
    # Fetch data for analysis period (first 7 days)
    analysis_data = get_latest_price(symbol, 'short', analysis_start, analysis_end)

    # Fetch full data including review period (all days)
    full_data = get_latest_price(symbol, 'short', analysis_start, review_end)

    # Fetch future data only (future day in the past)
    future_data = get_latest_price(symbol, 'short', analysis_end, review_end)
    return {
        'analysis_data': analysis_data,
        'analysis_future_data': future_data,
        'full_data': full_data,          # All 14 days for review
        'analysis_period': f"{analysis_start} → {analysis_end}",
        'review_period': f"{analysis_end} → {review_end}"
    }


# ----------------------------------------------------------------------
# Refactored backtesting workflow
# ----------------------------------------------------------------------
async def run_back_testing(analysis_start: str, analysis_end: str, review_end: str, symbol: str = 'btc') -> ReviewerAnalysis:
    """
    Run backtesting workflow with pre-fetched data for both periods.

    Args:
        analysis_start: Start date for analysis period (7 days)
        analysis_end: End date for analysis period / Start of review period
        review_end: End date for review period (next 7 days)
    """

    # --- 1. Fetch all data upfront ---
    print(f"\n{'='*60}")
    print(f"Fetching data: {analysis_start} → {review_end}")
    print(f"  Analysis period: {analysis_start} → {analysis_end}")
    print(f"  Review period: {analysis_end} → {review_end}")
    print(f"{'='*60}\n")

    data = fetch_backtest_data(analysis_start, analysis_end, review_end, symbol)

    # --- 2. Technical Analysis Agent (uses only first 7 days) ---
    tech_prompt = f"Phân tích kỹ thuật {symbol} dựa trên dữ liệu {data['analysis_data']}"

    tech_agent = Agent(
        name="technical-analysis-agent",
        model="gpt-5-mini",
        instructions=technical_analysis_prompt,
    )

    result = await Runner.run(tech_agent, tech_prompt)
    print(f"✓ Technical Analysis completed ({data['analysis_period']})")
    print(f"  Preview: {result.final_output[:200]}...\n")

    # --- 3. Reviewer Agent (uses full 14 days to compare predictions vs reality) ---
    review_context = {
        'analysis_period': data['analysis_period'],
        'review_period': data['review_period'],
        'technical_analysis': result.final_output,
        'analysis_data': data['analysis_data'],
        'full_data': data['full_data']
    }

    reviewer_user_prompt = f"""
    Đánh giá phân tích kỹ thuật sau đây trong bối cảnh backtesting:

    **Thời gian phân tích:** {data['analysis_period']}
    **Thời gian đánh giá (thực tế):** {data['review_period']}
    **Coin:** {symbol}

    **Phân tích kỹ thuật của agent:**
    {result.final_output}

    **Dữ liệu thực tế (full period để so sánh):**
    {data['full_data']}

    Hãy so sánh dự đoán với kết quả thực tế và đánh giá chất lượng phân tích.
    """

    reviewer_agent = Agent(
        name="reviewer-agent",
        model="gpt-5-mini",
        instructions=reviewer_prompt,
        tools=[
            WebSearchTool(),
            CodeInterpreterTool(
                tool_config={
                    "type": "code_interpreter",
                    "container": {"type": "auto"}
                }
            ),
        ],
    )

    review_result = await Runner.run(reviewer_agent, reviewer_user_prompt)
    print(f"✓ Review completed ({data['review_period']})")
    print(f"  Preview: {review_result.final_output[:200]}...\n")
    
    # --- 4. Crypto Trading Agent ---
    trading_agent = Agent(
        name="trading-agent",
        model="gpt-5-mini",
        instructions=trader_prompt,
    )
    
    trading_prompt = f"""
    Mô phỏng giao dịch BTC dựa trên phân tích kỹ thuật sau đây:
    {result.final_output}

    Diễn biến giá thực tế
    {data['analysis_future_data']}
    """
    trading_result = await Runner.run(trading_agent, trading_prompt)
    
    # --- 4. Create combined result ---
    report_analysis = ReviewerAnalysis(
        analysis_start=analysis_start,
        analysis_end=analysis_end,
        review_start=analysis_end,
        review_end=review_end,
        analysis=result.final_output,
        review=review_result.final_output,
        trading=trading_result.final_output,
        analysis_token=result.raw_responses[0].usage.total_tokens,
        review_token=review_result.raw_responses[0].usage.total_tokens,
    )

    return report_analysis

In [34]:
fetch_backtest_data_result = fetch_backtest_data('2025-10-01', '2025-10-09', '2025-10-12', 'eth')
fetch_backtest_data_result['analysis_future_data']

'{"eth":{"short":{"data":{"kline":[{"close_time":"2025-10-09 07:00:00","open_price":4515.28,"high_price":4539.31,"low_price":4500.18,"close_price":4525.72,"volume":24876.156},{"close_time":"2025-10-09 11:00:00","open_price":4525.72,"high_price":4531.52,"low_price":4398.17,"close_price":4450.8,"volume":95180.185},{"close_time":"2025-10-09 15:00:00","open_price":4450.8,"high_price":4460.0,"low_price":4411.0,"close_price":4426.6,"volume":53292.323},{"close_time":"2025-10-09 19:00:00","open_price":4426.6,"high_price":4426.69,"low_price":4320.0,"close_price":4373.05,"volume":135485.784},{"close_time":"2025-10-09 23:00:00","open_price":4373.06,"high_price":4413.76,"low_price":4291.82,"close_price":4314.79,"volume":119759.575},{"close_time":"2025-10-10 03:00:00","open_price":4314.78,"high_price":4344.33,"low_price":4265.06,"close_price":4338.0,"volume":77400.707},{"close_time":"2025-10-10 07:00:00","open_price":4338.03,"high_price":4383.0,"low_price":4327.89,"close_price":4368.09,"volume":348

In [21]:
import asyncio

# ----------------------------------------------------------------------
# Helper: Generate overlapping backtest windows (sliding window)
# ----------------------------------------------------------------------
def generate_backtest_windows(
    start_date: str,
    end_date: str,
    analysis_days: int = 7,
    review_days: int = 7,
    stride: int = 3
):
    """
    Generate overlapping backtest windows using sliding window approach.
    
    Args:
        start_date: Overall start date (e.g., '2025-06-01')
        end_date: Overall end date (e.g., '2025-11-03')
        analysis_days: Number of days for technical analysis (default: 7)
        review_days: Number of days for review period (default: 7)
        stride: Number of days to slide forward for each window (default: 3)
                This creates overlapping windows. Use stride=analysis_days+review_days for non-overlapping.
    
    Yields:
        Tuples of (analysis_start, analysis_end, review_end)
    
    Example:
        stride=3 with 7+7 day windows:
        - Window 1: 2025-06-01 → 2025-06-08 → 2025-06-15
        - Window 2: 2025-06-04 → 2025-06-11 → 2025-06-18 (slides 3 days)
        - Window 3: 2025-06-07 → 2025-06-14 → 2025-06-21 (slides 3 days)
    """
    from datetime import datetime, timedelta
    
    current = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    window_size = analysis_days + review_days
    
    while current < end:
        analysis_end = current + timedelta(days=analysis_days)
        review_end = current + timedelta(days=window_size)
        
        # Stop if we don't have enough data for review period
        if review_end > end:
            break
        
        yield (
            current.strftime("%Y-%m-%d"),           # analysis_start
            analysis_end.strftime("%Y-%m-%d"),      # analysis_end
            review_end.strftime("%Y-%m-%d")         # review_end
        )
        
        # Move to next window using stride (overlapping)
        current = current + timedelta(days=stride)


# ----------------------------------------------------------------------
# Main entry point (updated for sliding windows)
# ----------------------------------------------------------------------
async def entry_point():
    """Run backtesting across multiple overlapping time windows."""
    
    # Generate backtest windows (7 days analysis + 7 days review, sliding by 3 days)
    windows = list(generate_backtest_windows("2025-06-01", "2025-11-03", 7, 7, stride=3))
    
    print(f"\n{'='*60}")
    print(f"BACKTESTING PLAN (SLIDING WINDOW)")
    print(f"{'='*60}")
    print(f"Total windows: {len(windows)}")
    print(f"Window structure: 7 days analysis + 7 days review")
    print(f"Sliding stride: 3 days (overlapping windows)")
    print(f"\nFirst 5 windows:")
    for i, (a_start, a_end, r_end) in enumerate(windows[:5], 1):
        print(f"  {i}. Analysis: {a_start} → {a_end}, Review: {a_end} → {r_end}")
    if len(windows) > 5:
        print(f"  ... and {len(windows) - 5} more windows")
    print(f"{'='*60}\n")
    
    # Run all windows in parallel
    tasks = [run_back_testing(a_start, a_end, r_end) for a_start, a_end, r_end in windows]
    
    print("Starting parallel execution...\n")
    report: List[ReviewerAnalysis] = await asyncio.gather(*tasks)
    
    # Save results
    with open('report.json', 'w', encoding='utf-8') as f:
        json.dump([r.model_dump() for r in report], f, ensure_ascii=False, indent=2)
    
    print(f"\n{'='*60}")
    print(f"BACKTESTING COMPLETE")
    print(f"{'='*60}")
    print(f"Total reports generated: {len(report)}")
    print(f"Saved to: report.json")
    
    # Summary statistics
    total_analysis_tokens = sum(r.analysis_token for r in report)
    total_review_tokens = sum(r.review_token for r in report)
    print(f"\nToken usage:")
    print(f"  Analysis: {total_analysis_tokens:,}")
    print(f"  Review: {total_review_tokens:,}")
    print(f"  Total: {total_analysis_tokens + total_review_tokens:,}")
    print(f"{'='*60}\n")
    
    return report

# Run the backtesting
# asyncio.run(entry_point())

In [22]:
# Example: Test the new workflow with a single window
async def test_single_window():
    """Test backtesting with a single 7+3 day window."""
    
    # Test window: Oct 1-8 for analysis, Oct 8-15 for review
    result = await run_back_testing(
        analysis_start="2025-10-01",
        analysis_end="2025-10-08", 
        review_end="2025-10-11",
        symbol="eth"
    )
    
    print("\n" + "="*60)
    print("SINGLE WINDOW TEST RESULT")
    print("="*60)
    print(f"Analysis period: {result.analysis_start} → {result.analysis_end}")
    print(f"Review period: {result.review_start} → {result.review_end}")
    print(f"Analysis tokens: {result.analysis_token:,}")
    print(f"Review tokens: {result.review_token:,}")
    print("="*60 + "\n")
    
    return result

# Uncomment to test:
# asyncio.run(test_single_window())

In [23]:
result = await run_back_testing(
        analysis_start="2025-10-01",
        analysis_end="2025-10-08", 
        review_end="2025-10-11",
        symbol="eth"
    )


Fetching data: 2025-10-01 → 2025-10-11
  Analysis period: 2025-10-01 → 2025-10-08
  Review period: 2025-10-08 → 2025-10-11

✓ Technical Analysis completed (2025-10-01 → 2025-10-08)
  Preview: Dưới đây là phân tích kỹ thuật ETH trên khung 4H (giờ VN UTC+7) dựa trên dữ liệu bạn cung cấp (kline 4H đến 2025-10-09 03:00). Mình trình bày theo cấu trúc yêu cầu.

Tổng quan nhanh
- Giá hiện tại (cl...

✓ Review completed (2025-10-08 → 2025-10-11)
  Preview: Dưới đây là đánh giá backtesting (tiếng Việt) theo các tiêu chí bạn yêu cầu.

Tóm tắt ngắn (kết quả thực tế so với dự đoán)
- Điểm time0 của agent: close 4H gần nhất 2025-10-09 03:00 = 4,515.27 USD.  ...



In [24]:
print(result)

analysis_start='2025-10-01' analysis_end='2025-10-08' review_start='2025-10-08' review_end='2025-10-11' analysis='Dưới đây là phân tích kỹ thuật ETH trên khung 4H (giờ VN UTC+7) dựa trên dữ liệu bạn cung cấp (kline 4H đến 2025-10-09 03:00). Mình trình bày theo cấu trúc yêu cầu.\n\nTổng quan nhanh\n- Giá hiện tại (close 4H gần nhất): 4,515.27 USD (close_time 2025-10-09 03:00)\n- Giá mở cửa cây 4H hiện tại: 4,447.15 USD\n- Biên độ 24h: +0.80% (so với close 2025-10-08 03:00 = 4,479.44 → hiện tại 4,515.27)\n- Lưu ý: phân tích dựa trên dữ liệu 4H, chỉ ra các mức theo vùng ±0.5% khi cần.\n\n1) Diễn biến giá (4H gần nhất)\n- Hành động giá 4H gần nhất:\n  - Cây 4H cuối: open 4,447.15 → high 4,558.00 → low 4,439.00 → close 4,515.27. Đây là cây tăng mạnh với thân lớn và high phá vòm kháng cự ngắn hạn (breakout intraday), đóng cửa ở phần trên thân → cấu trúc ngắn hạn chuyển sang tích cực.\n  - Hành động 4H trước đó là dao động side-down, sau đó có một cú tăng mạnh (volume lớn) tạo đà phục hồi.\n-

In [22]:
with open('report.json', 'w', encoding='utf-8') as f:
    json.dump([r for r in report], f, ensure_ascii=False, indent=2)


In [31]:
# Visualize the backtest windows structure (updated for sliding window)
def visualize_windows(
    start_date: str,
    end_date: str,
    analysis_days: int = 7,
    review_days: int = 7,
    stride: int = 3
):
    """Show how the backtest windows will be structured with sliding window approach."""
    
    windows = list(generate_backtest_windows(start_date, end_date, analysis_days, review_days, stride))
    
    print(f"\n{'='*80}")
    print(f"BACKTEST WINDOWS STRUCTURE (SLIDING WINDOW)")
    print(f"{'='*80}")
    print(f"Period: {start_date} → {end_date}")
    print(f"Window size: {analysis_days} days analysis + {review_days} days review = {analysis_days + review_days} days total")
    print(f"Stride: {stride} days (sliding window - overlapping)")
    print(f"Total windows: {len(windows)}\n")
    
    # Show first 10 windows as example
    show_count = min(10, len(windows))
    for i, (a_start, a_end, r_end) in enumerate(windows[:show_count], 1):
        print(f"Window {i:2d}:")
        print(f"   Analysis: {a_start} → {a_end} (agent makes predictions)")
        print(f"   Review:   {a_end} → {r_end} (compare vs actual)")
        print()
    
    if len(windows) > show_count:
        print(f"... and {len(windows) - show_count} more windows\n")
    
    print(f"{'='*80}\n")
    
    # Calculate overlap
    if stride < analysis_days + review_days:
        overlap_days = (analysis_days + review_days) - stride
        print(f"Note: Windows overlap by {overlap_days} days (stride={stride}, window={analysis_days + review_days})")
        print(f"This provides more frequent analysis points and better temporal coverage.\n")
    
    return windows

# Example: Visualize the full backtest plan with sliding windows
# visualize_windows("2025-06-01", "2025-11-03", 7, 7, stride=3)

In [32]:
# ----------------------------------------------------------------------
# Token Cost Estimation for GPT-5-mini
# ----------------------------------------------------------------------
import tiktoken
from typing import Dict, Tuple

# GPT-5-mini pricing (per 1M tokens)
GPT5_MINI_INPUT_PRICE = 0.25  # USD per 1M input tokens
GPT5_MINI_OUTPUT_PRICE = 2.00  # USD per 1M output tokens

def count_tokens(text: str, model: str = "gpt-4o") -> int:
    """
    Count tokens in text using tiktoken.
    Uses gpt-4o encoding as proxy for gpt-5-mini (same tokenizer family).
    
    Args:
        text: Text to count tokens for
        model: Model name for encoding (default: gpt-4o)
    
    Returns:
        Token count
    """
    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        # Fallback to cl100k_base (used by GPT-4 and GPT-4o)
        encoding = tiktoken.get_encoding("cl100k_base")
    
    return len(encoding.encode(text))


def estimate_input_tokens(
    analysis_data: str,
    full_data: str,
    analysis_output: str = None
) -> Dict[str, int]:
    """
    Estimate input tokens for both technical analysis and reviewer agents.
    
    Args:
        analysis_data: JSON data for analysis period (7 days)
        full_data: JSON data for full period (14 days)
        analysis_output: Output from technical analysis (for reviewer input)
    
    Returns:
        Dict with token counts for each component
    """
    # Technical Analysis Input Tokens
    tech_prompt_tokens = count_tokens(technical_analysis_prompt)
    tech_user_prompt = f"Phân tích kỹ thuật BTC dựa trên dữ liệu {analysis_data}"
    tech_user_tokens = count_tokens(tech_user_prompt)
    tech_total_input = tech_prompt_tokens + tech_user_tokens
    
    # Reviewer Input Tokens (if analysis_output provided)
    reviewer_total_input = 0
    if analysis_output:
        reviewer_prompt_tokens = count_tokens(reviewer_prompt)
        reviewer_user_prompt = f"""
        Đánh giá phân tích kỹ thuật sau đây trong bối cảnh backtesting:

        **Phân tích kỹ thuật của agent:**
        {analysis_output}

        **Dữ liệu thực tế (full period để so sánh):**
        {full_data}
        """
        reviewer_user_tokens = count_tokens(reviewer_user_prompt)
        reviewer_total_input = reviewer_prompt_tokens + reviewer_user_tokens
    
    return {
        "tech_analysis_input": tech_total_input,
        "tech_prompt": tech_prompt_tokens,
        "tech_user": tech_user_tokens,
        "reviewer_input": reviewer_total_input,
        "reviewer_prompt": count_tokens(reviewer_prompt) if analysis_output else 0,
        "reviewer_user": reviewer_total_input - count_tokens(reviewer_prompt) if analysis_output else 0
    }


def estimate_output_tokens_from_sample(
    sample_analysis: str,
    sample_review: str
) -> Dict[str, int]:
    """
    Estimate output tokens based on sample outputs from report.json.
    
    Args:
        sample_analysis: Sample technical analysis output
        sample_review: Sample review output
    
    Returns:
        Dict with estimated output token counts
    """
    return {
        "tech_analysis_output": count_tokens(sample_analysis),
        "reviewer_output": count_tokens(sample_review)
    }


def calculate_cost(
    input_tokens: int,
    output_tokens: int,
    input_price_per_1m: float = GPT5_MINI_INPUT_PRICE,
    output_price_per_1m: float = GPT5_MINI_OUTPUT_PRICE
) -> Dict[str, float]:
    """
    Calculate cost for given token counts.
    
    Args:
        input_tokens: Number of input tokens
        output_tokens: Number of output tokens
        input_price_per_1m: Price per 1M input tokens
        output_price_per_1m: Price per 1M output tokens
    
    Returns:
        Dict with cost breakdown
    """
    input_cost = (input_tokens / 1_000_000) * input_price_per_1m
    output_cost = (output_tokens / 1_000_000) * output_price_per_1m
    total_cost = input_cost + output_cost
    
    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
        "input_cost_usd": round(input_cost, 6),
        "output_cost_usd": round(output_cost, 6),
        "total_cost_usd": round(total_cost, 6)
    }


def estimate_window_cost(
    analysis_start: str,
    analysis_end: str,
    review_end: str,
    avg_analysis_output_tokens: int = 2500,  # Estimated from sample
    avg_review_output_tokens: int = 6000     # Estimated from sample
) -> Dict[str, any]:
    """
    Estimate cost for a single backtest window.
    
    Args:
        analysis_start: Start date for analysis
        analysis_end: End date for analysis
        review_end: End date for review
        avg_analysis_output_tokens: Average tokens in analysis output
        avg_review_output_tokens: Average tokens in review output
    
    Returns:
        Dict with detailed cost estimate
    """
    # Fetch data to calculate input tokens
    data = fetch_backtest_data(analysis_start, analysis_end, review_end)
    
    # Estimate input tokens (without analysis output first)
    input_est = estimate_input_tokens(data['analysis_data'], data['full_data'])
    
    # Technical Analysis Cost
    tech_cost = calculate_cost(
        input_est['tech_analysis_input'],
        avg_analysis_output_tokens
    )
    
    # Reviewer Cost (estimate input based on average analysis output)
    # Re-calculate with estimated analysis output for more accurate reviewer input
    dummy_analysis = "x" * (avg_analysis_output_tokens * 4)  # ~4 chars per token
    reviewer_input_est = estimate_input_tokens(
        data['analysis_data'],
        data['full_data'],
        dummy_analysis
    )
    
    reviewer_cost = calculate_cost(
        reviewer_input_est['reviewer_input'],
        avg_review_output_tokens
    )
    
    # Total for window
    total_cost = tech_cost['total_cost_usd'] + reviewer_cost['total_cost_usd']
    total_tokens = tech_cost['total_tokens'] + reviewer_cost['total_tokens']
    
    return {
        "window": f"{analysis_start} → {analysis_end} → {review_end}",
        "technical_analysis": tech_cost,
        "reviewer": reviewer_cost,
        "total_cost_usd": round(total_cost, 6),
        "total_tokens": total_tokens
    }


def estimate_full_backtest_cost(
    start_date: str,
    end_date: str,
    analysis_days: int = 7,
    review_days: int = 7,
    stride: int = 3
) -> Dict[str, any]:
    """
    Estimate total cost for full backtest run with sliding windows.
    
    Args:
        start_date: Overall start date
        end_date: Overall end date
        analysis_days: Days for analysis period
        review_days: Days for review period
        stride: Days to slide forward for each window (default: 3)
    
    Returns:
        Dict with detailed cost breakdown and summary
    """
    windows = list(generate_backtest_windows(start_date, end_date, analysis_days, review_days, stride))
    
    print(f"\n{'='*80}")
    print(f"COST ESTIMATION FOR BACKTEST (SLIDING WINDOW)")
    print(f"{'='*80}")
    print(f"Period: {start_date} → {end_date}")
    print(f"Total windows: {len(windows)}")
    print(f"Stride: {stride} days (overlapping windows)")
    print(f"Pricing: GPT-5-mini")
    print(f"  - Input: ${GPT5_MINI_INPUT_PRICE} per 1M tokens")
    print(f"  - Output: ${GPT5_MINI_OUTPUT_PRICE} per 1M tokens")
    print(f"{'='*80}\n")
    
    # Estimate first window in detail
    first_window = windows[0]
    sample_estimate = estimate_window_cost(*first_window)
    
    print(f"Sample Window Estimate (Window 1):")
    print(f"  Period: {sample_estimate['window']}")
    print(f"  Technical Analysis:")
    print(f"    - Input tokens: {sample_estimate['technical_analysis']['input_tokens']:,}")
    print(f"    - Output tokens: {sample_estimate['technical_analysis']['output_tokens']:,}")
    print(f"    - Cost: ${sample_estimate['technical_analysis']['total_cost_usd']:.6f}")
    print(f"  Reviewer:")
    print(f"    - Input tokens: {sample_estimate['reviewer']['input_tokens']:,}")
    print(f"    - Output tokens: {sample_estimate['reviewer']['output_tokens']:,}")
    print(f"    - Cost: ${sample_estimate['reviewer']['total_cost_usd']:.6f}")
    print(f"  Window Total: ${sample_estimate['total_cost_usd']:.6f}\n")
    
    # Extrapolate to all windows
    estimated_total_cost = sample_estimate['total_cost_usd'] * len(windows)
    estimated_total_tokens = sample_estimate['total_tokens'] * len(windows)
    
    print(f"{'='*80}")
    print(f"TOTAL ESTIMATED COST")
    print(f"{'='*80}")
    print(f"Total windows: {len(windows)}")
    print(f"Cost per window: ${sample_estimate['total_cost_usd']:.6f}")
    print(f"Total tokens: {estimated_total_tokens:,}")
    print(f"Total cost: ${estimated_total_cost:.2f}")
    print(f"{'='*80}\n")
    
    return {
        "windows_count": len(windows),
        "sample_window": sample_estimate,
        "cost_per_window": sample_estimate['total_cost_usd'],
        "tokens_per_window": sample_estimate['total_tokens'],
        "estimated_total_cost": round(estimated_total_cost, 2),
        "estimated_total_tokens": estimated_total_tokens,
        "stride": stride
    }

In [ ]:
# ----------------------------------------------------------------------
# Enhanced Backtest Execution with PARALLEL Progressive Saving
# ----------------------------------------------------------------------

class BacktestReport(BaseModel):
    """Extended report with cost tracking"""
    analysis_start: str
    analysis_end: str
    review_start: str
    review_end: str
    analysis: str
    review: str
    analysis_token: int
    review_token: int
    # Cost breakdown
    analysis_input_tokens: int
    analysis_output_tokens: int
    review_input_tokens: int
    review_output_tokens: int
    analysis_cost_usd: float
    review_cost_usd: float
    total_cost_usd: float
    # Status tracking
    status: str = "completed"  # "completed" or "failed"
    error_message: str = None


async def run_backtest_with_cost_tracking(
    analysis_start: str,
    analysis_end: str,
    review_end: str
) -> BacktestReport:
    """
    Run backtesting workflow with detailed cost tracking and error handling.
    
    Args:
        analysis_start: Start date for analysis period (7 days)
        analysis_end: End date for analysis period / Start of review period
        review_end: End date for review period (next 7 days)
    
    Returns:
        BacktestReport with cost information
    """
    try:
        # --- 1. Fetch all data upfront ---
        print(f"\n{'='*60}")
        print(f"Fetching data: {analysis_start} → {review_end}")
        print(f"  Analysis period: {analysis_start} → {analysis_end}")
        print(f"  Review period: {analysis_end} → {review_end}")
        print(f"{'='*60}\n")

        data = fetch_backtest_data(analysis_start, analysis_end, review_end)

        # --- 2. Technical Analysis Agent ---
        tech_prompt = f"Phân tích kỹ thuật BTC dựa trên dữ liệu {data['analysis_data']}"
        
        # Calculate input tokens
        tech_input_tokens = count_tokens(technical_analysis_prompt) + count_tokens(tech_prompt)

        tech_agent = Agent(
            name="technical-analysis-agent",
            model="gpt-5-mini",
            instructions=technical_analysis_prompt,
        )

        result = await Runner.run(tech_agent, tech_prompt)
        print(f"✓ Technical Analysis completed ({data['analysis_period']})")
        
        # Calculate output tokens
        tech_output_tokens = count_tokens(result.final_output)
        tech_cost = calculate_cost(tech_input_tokens, tech_output_tokens)
        
        print(f"  Tokens: {tech_cost['input_tokens']:,} in + {tech_cost['output_tokens']:,} out = {tech_cost['total_tokens']:,}")
        print(f"  Cost: ${tech_cost['total_cost_usd']:.6f}\n")

        # --- 3. Reviewer Agent ---
        reviewer_user_prompt = f"""
        Đánh giá phân tích kỹ thuật sau đây trong bối cảnh backtesting:

        **Thời gian phân tích:** {data['analysis_period']}
        **Thời gian đánh giá (thực tế):** {data['review_period']}

        **Phân tích kỹ thuật của agent:**
        {result.final_output}

        **Dữ liệu thực tế (full period để so sánh):**
        {data['full_data']}

        Hãy so sánh dự đoán với kết quả thực tế và đánh giá chất lượng phân tích.
        """
        
        # Calculate input tokens
        review_input_tokens = count_tokens(reviewer_prompt) + count_tokens(reviewer_user_prompt)

        reviewer_agent = Agent(
            name="reviewer-agent",
            model="gpt-5-mini",
            instructions=reviewer_prompt,
            tools=[
                WebSearchTool(),
                CodeInterpreterTool(
                    tool_config={
                        "type": "code_interpreter",
                        "container": {"type": "auto"}
                    }
                ),
            ],
        )

        review_result = await Runner.run(reviewer_agent, reviewer_user_prompt)
        print(f"✓ Review completed ({data['review_period']})")
        
        # Calculate output tokens
        review_output_tokens = count_tokens(review_result.final_output)
        review_cost = calculate_cost(review_input_tokens, review_output_tokens)
        
        print(f"  Tokens: {review_cost['input_tokens']:,} in + {review_cost['output_tokens']:,} out = {review_cost['total_tokens']:,}")
        print(f"  Cost: ${review_cost['total_cost_usd']:.6f}\n")

        # --- 4. Create report with cost tracking ---
        total_cost = tech_cost['total_cost_usd'] + review_cost['total_cost_usd']
        
        print(f"{'─'*60}")
        print(f"Window Total: ${total_cost:.6f}")
        print(f"{'─'*60}\n")

        report = BacktestReport(
            analysis_start=analysis_start,
            analysis_end=analysis_end,
            review_start=analysis_end,
            review_end=review_end,
            analysis=result.final_output,
            review=review_result.final_output,
            analysis_token=result.raw_responses[0].usage.total_tokens,
            review_token=review_result.raw_responses[0].usage.total_tokens,
            analysis_input_tokens=tech_input_tokens,
            analysis_output_tokens=tech_output_tokens,
            review_input_tokens=review_input_tokens,
            review_output_tokens=review_output_tokens,
            analysis_cost_usd=round(tech_cost['total_cost_usd'], 6),
            review_cost_usd=round(review_cost['total_cost_usd'], 6),
            total_cost_usd=round(total_cost, 6),
            status="completed"
        )

        return report
        
    except Exception as e:
        # Return failed report with error details
        print(f"\n❌ Error in window {analysis_start} → {review_end}")
        print(f"   Error: {str(e)[:200]}\n")
        
        return BacktestReport(
            analysis_start=analysis_start,
            analysis_end=analysis_end,
            review_start=analysis_end,
            review_end=review_end,
            analysis="",
            review="",
            analysis_token=0,
            review_token=0,
            analysis_input_tokens=0,
            analysis_output_tokens=0,
            review_input_tokens=0,
            review_output_tokens=0,
            analysis_cost_usd=0.0,
            review_cost_usd=0.0,
            total_cost_usd=0.0,
            status="failed",
            error_message=str(e)
        )


def save_progress(reports: List[BacktestReport], output_file: str):
    """Save current progress to file"""
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump([r.model_dump() for r in reports], f, ensure_ascii=False, indent=2)


def load_existing_results(output_file: str) -> List[BacktestReport]:
    """Load existing results if file exists"""
    try:
        with open(output_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return [BacktestReport(**item) for item in data]
    except FileNotFoundError:
        return []


async def execute_full_backtest(
    start_date: str,
    end_date: str,
    analysis_days: int = 7,
    review_days: int = 7,
    stride: int = 3,
    parallel: bool = False,
    output_file: str = 'backtest_report.json',
    resume: bool = True,
    save_every_n: int = 1  # Save after every N completions in parallel mode
) -> List[BacktestReport]:
    """
    Execute full backtest with progressive saving (works in BOTH parallel and sequential).
    
    Args:
        start_date: Overall start date (e.g., '2025-06-01')
        end_date: Overall end date (e.g., '2025-11-03')
        analysis_days: Number of days for technical analysis (default: 7)
        review_days: Number of days for review period (default: 7)
        stride: Days to slide forward for each window (default: 3)
        parallel: Run windows in parallel with progressive saving (default: False)
        output_file: Output JSON file path (default: 'backtest_report.json')
        resume: Resume from existing results (default: True)
        save_every_n: In parallel mode, save after every N completions (default: 1)
    
    Returns:
        List of BacktestReport objects
    """
    # Generate backtest windows with sliding window
    windows = list(generate_backtest_windows(start_date, end_date, analysis_days, review_days, stride))
    
    overlap_days = (analysis_days + review_days) - stride if stride < (analysis_days + review_days) else 0
    
    print(f"\n{'='*80}")
    print(f"EXECUTING BACKTEST (SLIDING WINDOW)")
    print(f"{'='*80}")
    print(f"Period: {start_date} → {end_date}")
    print(f"Total windows: {len(windows)}")
    print(f"Window structure: {analysis_days} days analysis + {review_days} days review")
    print(f"Stride: {stride} days ({'overlapping by ' + str(overlap_days) + ' days' if overlap_days > 0 else 'non-overlapping'})")
    print(f"Execution mode: {'Parallel with progressive saving' if parallel else 'Sequential with progressive saving'}")
    print(f"Output file: {output_file}")
    print(f"Resume mode: {'Enabled' if resume else 'Disabled'}")
    if parallel:
        print(f"Save frequency: Every {save_every_n} completion(s)")
    print(f"{'='*80}\n")
    
    # Load existing results if resume enabled
    existing_reports = []
    completed_windows = set()
    
    if resume:
        existing_reports = load_existing_results(output_file)
        completed_windows = {
            (r.analysis_start, r.analysis_end, r.review_end) 
            for r in existing_reports if r.status == "completed"
        }
        if completed_windows:
            print(f"📂 Loaded {len(completed_windows)} completed windows from {output_file}")
            print(f"   Skipping already completed windows...\n")
    
    # Filter windows to process
    windows_to_process = [
        w for w in windows 
        if w not in completed_windows
    ]
    
    if not windows_to_process:
        print("✅ All windows already completed!")
        return existing_reports
    
    print(f"📋 Processing {len(windows_to_process)} windows ({len(completed_windows)} already done)\n")
    
    # Run backtest
    if parallel:
        print(f"🚀 Starting parallel execution of {len(windows_to_process)} windows...")
        print(f"   ⚡ Progressive saving enabled - saves every {save_every_n} completion(s)\n")
        
        all_reports = existing_reports.copy()
        completed_count = 0
        
        # Create all tasks
        tasks = {
            asyncio.create_task(run_backtest_with_cost_tracking(a_start, a_end, r_end)): (a_start, a_end, r_end)
            for a_start, a_end, r_end in windows_to_process
        }
        
        # Process as they complete
        for coro in asyncio.as_completed(tasks.keys()):
            try:
                report = await coro
                completed_count += 1
                
                # Add to results
                all_reports.append(report)
                
                # Log completion
                if report.status == "completed":
                    print(f"✅ [{completed_count}/{len(windows_to_process)}] {report.analysis_start} → {report.review_end} (${report.total_cost_usd:.4f})")
                else:
                    print(f"❌ [{completed_count}/{len(windows_to_process)}] {report.analysis_start} → {report.review_end} - {report.error_message[:80]}")
                
                # Save progress
                if completed_count % save_every_n == 0 or completed_count == len(windows_to_process):
                    save_progress(all_reports, output_file)
                    print(f"   💾 Progress saved ({len(all_reports)}/{len(windows)} total)\n")
                    
            except Exception as e:
                # This shouldn't happen since we catch exceptions in run_backtest_with_cost_tracking
                print(f"⚠️  Unexpected error: {str(e)[:100]}")
                completed_count += 1
        
    else:
        print(f"Starting sequential execution with progressive saving...\n")
        all_reports = existing_reports.copy()
        
        for i, (a_start, a_end, r_end) in enumerate(windows_to_process, 1):
            print(f"{'='*80}")
            print(f"Window {len(completed_windows) + i}/{len(windows)}: {a_start} → {a_end} → {r_end}")
            print(f"{'='*80}")
            
            report = await run_backtest_with_cost_tracking(a_start, a_end, r_end)
            all_reports.append(report)
            
            # Save progress after each window
            save_progress(all_reports, output_file)
            print(f"💾 Progress saved to {output_file} ({len(all_reports)}/{len(windows)} windows)\n")
    
    # Summary statistics
    completed_reports = [r for r in all_reports if r.status == "completed"]
    failed_reports = [r for r in all_reports if r.status == "failed"]
    
    total_analysis_tokens = sum(r.analysis_token for r in completed_reports)
    total_review_tokens = sum(r.review_token for r in completed_reports)
    total_tokens = total_analysis_tokens + total_review_tokens
    total_cost = sum(r.total_cost_usd for r in completed_reports)
    
    print(f"\n{'='*80}")
    print(f"BACKTEST COMPLETE")
    print(f"{'='*80}")
    print(f"Total windows: {len(all_reports)}")
    print(f"  ✅ Completed: {len(completed_reports)}")
    print(f"  ❌ Failed: {len(failed_reports)}")
    print(f"Output saved to: {output_file}")
    
    if failed_reports:
        print(f"\n⚠️  Failed Windows:")
        for r in failed_reports[:10]:  # Show first 10
            print(f"  - {r.analysis_start} → {r.review_end}")
            print(f"    Error: {r.error_message[:100]}...")
        if len(failed_reports) > 10:
            print(f"  ... and {len(failed_reports)-10} more")
    
    print(f"\nToken Usage (Completed Windows):")
    print(f"  Technical Analysis: {total_analysis_tokens:,}")
    print(f"  Review: {total_review_tokens:,}")
    print(f"  Total: {total_tokens:,}")
    print(f"\nCost Breakdown (GPT-5-mini):")
    print(f"  Input: ${GPT5_MINI_INPUT_PRICE}/1M tokens")
    print(f"  Output: ${GPT5_MINI_OUTPUT_PRICE}/1M tokens")
    print(f"\n  Total Cost: ${total_cost:.2f}")
    if completed_reports:
        print(f"  Average per window: ${total_cost/len(completed_reports):.4f}")
    print(f"{'='*80}\n")
    
    return all_reports

In [ ]:
# ----------------------------------------------------------------------
# USAGE EXAMPLES (Updated for Sliding Window)
# ----------------------------------------------------------------------

# Example 1: Estimate cost before running backtest with sliding windows
async def estimate_cost_example():
    """Example: Estimate cost for the full backtest period with stride=3"""
    cost_estimate = estimate_full_backtest_cost(
        start_date="2025-06-01",
        end_date="2025-11-03",
        analysis_days=7,
        review_days=7,
        stride=3  # Sliding window: overlap by 11 days
    )
    return cost_estimate

# Example 2: Run full backtest with cost tracking and sliding windows
async def run_full_backtest_example():
    """Example: Execute full backtest with cost tracking and stride=3"""
    reports = await execute_full_backtest(
        start_date="2025-06-01",
        end_date="2025-11-03",
        analysis_days=7,
        review_days=7,
        stride=3,  # Slide by 3 days
        parallel=True,  # Set to False for sequential execution
        output_file="backtest_report_full.json"
    )
    return reports

# Example 3: Run a single window test with cost tracking
async def test_single_window_with_cost():
    """Example: Test single window with cost tracking"""
    report = await run_backtest_with_cost_tracking(
        analysis_start="2025-10-01",
        analysis_end="2025-10-08",
        review_end="2025-10-15"
    )
    
    print("\n" + "="*60)
    print("SINGLE WINDOW COST ANALYSIS")
    print("="*60)
    print(f"Period: {report.analysis_start} → {report.review_end}")
    print(f"\nTechnical Analysis:")
    print(f"  Input tokens: {report.analysis_input_tokens:,}")
    print(f"  Output tokens: {report.analysis_output_tokens:,}")
    print(f"  Cost: ${report.analysis_cost_usd:.6f}")
    print(f"\nReviewer:")
    print(f"  Input tokens: {report.review_input_tokens:,}")
    print(f"  Output tokens: {report.review_output_tokens:,}")
    print(f"  Cost: ${report.review_cost_usd:.6f}")
    print(f"\nTotal:")
    print(f"  Total tokens: {report.analysis_token + report.review_token:,}")
    print(f"  Total cost: ${report.total_cost_usd:.6f}")
    print("="*60 + "\n")
    
    return report

# Example 4: Visualize windows with different stride values
def visualize_different_strides():
    """Example: Compare different stride configurations"""
    print("\n" + "="*80)
    print("COMPARING DIFFERENT STRIDE CONFIGURATIONS")
    print("="*80 + "\n")
    
    configs = [
        (3, "High overlap - more data points"),
        (7, "Medium overlap - balanced"),
        (14, "No overlap - non-overlapping windows")
    ]
    
    for stride, description in configs:
        windows = list(generate_backtest_windows("2025-06-01", "2025-11-03", 7, 7, stride))
        overlap = 14 - stride
        print(f"Stride = {stride} days ({description})")
        print(f"  - Total windows: {len(windows)}")
        print(f"  - Overlap: {overlap} days")
        print(f"  - First 3 windows:")
        for i, (a_start, a_end, r_end) in enumerate(windows[:3], 1):
            print(f"    {i}. {a_start} → {a_end} → {r_end}")
        print()

# Example 5: Analyze actual token usage from existing report.json
def analyze_existing_report():
    """Analyze token usage from existing report.json"""
    with open('report.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Get the first report (it's a list with single item containing key-value pairs)
    if isinstance(data[0], list):
        report_dict = dict(data[0])
    else:
        report_dict = data[0]
    
    analysis_text = report_dict['analysis']
    review_text = report_dict['review']
    
    # Count tokens
    output_tokens = estimate_output_tokens_from_sample(analysis_text, review_text)
    
    print("\n" + "="*80)
    print("ACTUAL TOKEN ANALYSIS FROM report.json")
    print("="*80)
    print(f"Analysis output tokens: {output_tokens['tech_analysis_output']:,}")
    print(f"Review output tokens: {output_tokens['reviewer_output']:,}")
    print(f"Total output tokens: {sum(output_tokens.values()):,}")
    print(f"\nReported by API:")
    print(f"  Analysis total: {report_dict['analysis_token']:,}")
    print(f"  Review total: {report_dict['review_token']:,}")
    print("="*80 + "\n")
    
    return output_tokens

# ----------------------------------------------------------------------
# Quick reference guide
# ----------------------------------------------------------------------
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                    BACKTEST EXECUTION GUIDE (SLIDING WINDOW)                 ║
╚══════════════════════════════════════════════════════════════════════════════╝

STEP 1: Visualize Windows
--------------------------
visualize_windows("2025-06-01", "2025-11-03", 7, 7, stride=3)

STEP 2: Estimate Cost
---------------------
await estimate_cost_example()

STEP 3: Test Single Window (Optional)
--------------------------------------
await test_single_window_with_cost()

STEP 4: Execute Full Backtest
------------------------------
reports = await execute_full_backtest(
    start_date="2025-06-01",
    end_date="2025-11-03",
    analysis_days=7,
    review_days=7,
    stride=3,           # 3-day sliding window
    parallel=True,
    output_file="backtest_report.json"
)

CONFIGURATION OPTIONS:
- stride=3  : High overlap (11 days), more data points (~52 windows for 5 months)
- stride=7  : Medium overlap (7 days), balanced (~26 windows)
- stride=14 : No overlap, non-overlapping windows (~11 windows)

""")

# Uncomment to run:
# await estimate_cost_example()
# await test_single_window_with_cost()
# await run_full_backtest_example()
# visualize_different_strides()
# analyze_existing_report()

In [ ]:
# ----------------------------------------------------------------------
# DEMO: Visualize sliding window effect
# ----------------------------------------------------------------------
def demo_sliding_window():
    """Visual demonstration of how sliding windows work"""
    print("\n" + "="*80)
    print("SLIDING WINDOW DEMONSTRATION (stride=3 vs stride=14)")
    print("="*80 + "\n")
    
    # Show first few windows with stride=3
    windows_overlap = list(generate_backtest_windows("2025-06-01", "2025-06-22", 7, 7, stride=3))
    
    print("WITH STRIDE=3 (Overlapping - 11 days overlap):")
    print("-" * 80)
    for i, (a_start, a_end, r_end) in enumerate(windows_overlap, 1):
        print(f"Window {i}: Analysis [{a_start} → {a_end}] + Review [{a_end} → {r_end}]")
    
    print("\n" + "-" * 80)
    print(f"Total windows: {len(windows_overlap)}")
    print(f"Coverage: {windows_overlap[0][0]} to {windows_overlap[-1][2]}")
    print(f"Windows overlap by 11 days - provides more frequent analysis points")
    
    # Show comparison with stride=14
    windows_non_overlap = list(generate_backtest_windows("2025-06-01", "2025-06-22", 7, 7, stride=14))
    
    print("\n\nWITH STRIDE=14 (Non-overlapping):")
    print("-" * 80)
    for i, (a_start, a_end, r_end) in enumerate(windows_non_overlap, 1):
        print(f"Window {i}: Analysis [{a_start} → {a_end}] + Review [{a_end} → {r_end}]")
    
    print("\n" + "-" * 80)
    print(f"Total windows: {len(windows_non_overlap)}")
    print(f"Coverage: {windows_non_overlap[0][0]} to {windows_non_overlap[-1][2]}")
    print(f"No overlap - each time period analyzed once")
    
    print("\n" + "="*80)
    print("RECOMMENDATION: Use stride=3 for better temporal coverage and more training data")
    print("="*80 + "\n")

# Run demo (uncomment to see)
# demo_sliding_window()

In [ ]:
# ----------------------------------------------------------------------
# EMERGENCY: Extract partial results from failed parallel execution
# ----------------------------------------------------------------------

def extract_partial_results_from_exception():
    """
    If asyncio.gather() failed but some tasks completed, 
    the results might still be accessible.
    
    Run this immediately after a failed parallel execution!
    """
    import sys
    
    # Try to get the last exception
    exc_type, exc_value, exc_traceback = sys.exc_info()
    
    print("Checking for partial results in memory...\n")
    
    # Check if there are any reports in local variables
    # In Jupyter, variables from the failed cell might still exist
    try:
        # Try to access 'reports' variable if it exists
        if 'reports' in globals():
            partial_reports = globals()['reports']
            print(f"✅ Found 'reports' variable in memory!")
            print(f"   Type: {type(partial_reports)}")
            print(f"   Length: {len(partial_reports) if hasattr(partial_reports, '__len__') else 'N/A'}")
            
            # Try to save whatever we have
            valid_reports = []
            for i, r in enumerate(partial_reports):
                if isinstance(r, BacktestReport):
                    valid_reports.append(r)
                    print(f"   [{i}] ✅ Valid report: {r.analysis_start} → {r.review_end}")
                elif isinstance(r, Exception):
                    print(f"   [{i}] ❌ Exception: {str(r)[:80]}")
                else:
                    print(f"   [{i}] ⚠️  Unknown type: {type(r)}")
            
            if valid_reports:
                # Save the valid reports
                output_file = "recovered_reports.json"
                save_progress(valid_reports, output_file)
                print(f"\n💾 Saved {len(valid_reports)} recovered reports to {output_file}")
                return valid_reports
        else:
            print("❌ No 'reports' variable found in global scope")
            
    except Exception as e:
        print(f"❌ Error extracting results: {e}")
    
    return None


def save_variables_to_json(var_name: str = "reports", output_file: str = "emergency_save.json"):
    """
    Manually save any variable to JSON
    
    Usage:
        save_variables_to_json("reports", "my_backup.json")
    """
    try:
        if var_name in globals():
            data = globals()[var_name]
            
            # If it's a list of BacktestReport objects
            if isinstance(data, list) and data and isinstance(data[0], BacktestReport):
                with open(output_file, 'w', encoding='utf-8') as f:
                    json.dump([r.model_dump() for r in data], f, ensure_ascii=False, indent=2)
                print(f"✅ Saved {len(data)} items from '{var_name}' to {output_file}")
                return True
            
            # If it's raw data
            else:
                with open(output_file, 'w', encoding='utf-8') as f:
                    json.dump(data, f, ensure_ascii=False, indent=2, default=str)
                print(f"✅ Saved '{var_name}' to {output_file}")
                return True
        else:
            print(f"❌ Variable '{var_name}' not found in current scope")
            available = [v for v in globals().keys() if not v.startswith('_')]
            print(f"Available variables: {available[:10]}...")  # Show first 10
            return False
            
    except Exception as e:
        print(f"❌ Error saving: {e}")
        return False


def list_available_variables():
    """List all variables in current Jupyter session"""
    print("\n" + "="*80)
    print("AVAILABLE VARIABLES IN CURRENT SESSION")
    print("="*80 + "\n")
    
    # Get all variables
    all_vars = {k: v for k, v in globals().items() if not k.startswith('_')}
    
    # Categorize
    reports_vars = {k: v for k, v in all_vars.items() if 'report' in k.lower()}
    data_vars = {k: v for k, v in all_vars.items() if 'data' in k.lower() or 'result' in k.lower()}
    
    if reports_vars:
        print("📊 Report-related variables:")
        for name, value in reports_vars.items():
            print(f"   {name}: {type(value).__name__}", end="")
            if hasattr(value, '__len__'):
                print(f" (length: {len(value)})")
            else:
                print()
    
    if data_vars:
        print("\n📦 Data-related variables:")
        for name, value in data_vars.items():
            print(f"   {name}: {type(value).__name__}", end="")
            if hasattr(value, '__len__'):
                print(f" (length: {len(value)})")
            else:
                print()
    
    print("\n" + "="*80 + "\n")
    
    return {
        'all': list(all_vars.keys()),
        'reports': list(reports_vars.keys()),
        'data': list(data_vars.keys())
    }


# Quick recovery commands
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                    EMERGENCY RECOVERY COMMANDS                               ║
╚══════════════════════════════════════════════════════════════════════════════╝

If your parallel execution failed but requests completed:

1. List available variables:
   >>> list_available_variables()

2. Try automatic extraction:
   >>> extract_partial_results_from_exception()

3. Manually save a variable:
   >>> save_variables_to_json("reports", "my_backup.json")
   >>> save_variables_to_json("your_variable_name", "backup.json")

4. Check specific variable:
   >>> print(type(reports))
   >>> print(len(reports))
   >>> reports[0]  # Check first item

""")

# Run this immediately after error!
# extract_partial_results_from_exception()

In [48]:
# ----------------------------------------------------------------------
# BATCH PREVIEW: Check what will be sent before submitting
# ----------------------------------------------------------------------

def preview_batch_requests(
    start_date: str,
    end_date: str,
    analysis_days: int = 7,
    review_days: int = 7,
    stride: int = 3,
    show_full_prompts: bool = False,
    max_windows_to_show: int = 3
):
    """
    Preview what will be sent in batch requests WITHOUT actually sending them.
    This helps identify prompts that might be flagged by content policy.
    
    Args:
        start_date: Overall start date
        end_date: Overall end date
        analysis_days: Days for analysis period
        review_days: Days for review period
        stride: Days to slide forward
        show_full_prompts: Show complete prompts (can be very long)
        max_windows_to_show: Number of windows to show in detail
    """
    print(f"\n{'='*80}")
    print(f"BATCH REQUESTS PREVIEW - What will be sent to OpenAI")
    print(f"{'='*80}\n")
    
    # Generate windows
    windows = list(generate_backtest_windows(start_date, end_date, analysis_days, review_days, stride))
    print(f"Total windows to process: {len(windows)}\n")
    
    # Analyze first few windows in detail
    print(f"{'='*80}")
    print(f"DETAILED ANALYSIS (first {max_windows_to_show} windows)")
    print(f"{'='*80}\n")
    
    total_tech_tokens = 0
    total_review_tokens = 0
    flagged_windows = []
    
    for i, (a_start, a_end, r_end) in enumerate(windows[:max_windows_to_show]):
        print(f"\n{'─'*80}")
        print(f"Window {i+1}: {a_start} → {a_end} → {r_end}")
        print(f"{'─'*80}")
        
        # Fetch data for this window
        try:
            data = fetch_backtest_data(a_start, a_end, r_end)
            
            # Technical Analysis Request
            tech_user_prompt = f"Phân tích kỹ thuật BTC dựa trên dữ liệu {data['analysis_data']}"
            tech_total = technical_analysis_prompt + tech_user_prompt
            tech_tokens = count_tokens(tech_total)
            total_tech_tokens += tech_tokens
            
            print(f"\n📊 TECHNICAL ANALYSIS REQUEST:")
            print(f"   System prompt: {len(technical_analysis_prompt):,} chars")
            print(f"   User prompt: {len(tech_user_prompt):,} chars")
            print(f"   Total tokens: ~{tech_tokens:,}")
            
            # Check for potential issues
            flags = []
            
            # Check for suspicious keywords
            suspicious = ['hack', 'exploit', 'scam', 'pump', 'dump', 'manipulate', 'insider', 'illegal', 'fraud']
            found = [kw for kw in suspicious if kw in tech_total.lower()]
            if found:
                flags.append(f"⚠️  Suspicious keywords: {found}")
            
            # Check trading signal density
            signals = ['long', 'short', 'buy', 'sell', 'tp', 'sl']
            signal_count = sum(tech_total.lower().count(kw) for kw in signals)
            if signal_count > 30:
                flags.append(f"⚠️  High signal density: {signal_count} occurrences")
            
            # Review Request (we'll generate this after getting tech analysis, but preview the structure)
            print(f"\n📝 REVIEW REQUEST (will be created after tech analysis):")
            print(f"   System prompt: {len(reviewer_prompt):,} chars")
            print(f"   User prompt: Will include tech analysis + review data")
            print(f"   Estimated tokens: ~3000-5000")
            
            if flags:
                print(f"\n🚨 POTENTIAL ISSUES:")
                for flag in flags:
                    print(f"   {flag}")
                flagged_windows.append((i+1, a_start, flags))
            else:
                print(f"\n✅ No obvious issues detected")
            
            if show_full_prompts:
                print(f"\n{'─'*40}")
                print(f"FULL TECHNICAL ANALYSIS PROMPT:")
                print(f"{'─'*40}")
                print(f"System: {technical_analysis_prompt[:200]}...")
                print(f"User: {tech_user_prompt[:500]}...")
                
        except Exception as e:
            print(f"\n❌ Error fetching data: {str(e)}")
            flagged_windows.append((i+1, a_start, [f"Data fetch error: {str(e)}"]))
    
    # Summary
    print(f"\n{'='*80}")
    print(f"PREVIEW SUMMARY")
    print(f"{'='*80}")
    print(f"Total windows: {len(windows)}")
    print(f"Previewed in detail: {min(max_windows_to_show, len(windows))}")
    print(f"Windows with potential issues: {len(flagged_windows)}")
    
    if flagged_windows:
        print(f"\n⚠️  FLAGGED WINDOWS:")
        for win_num, date, flags in flagged_windows:
            print(f"   Window {win_num} ({date}): {', '.join(flags)}")
        print(f"\n⚠️  Review these windows before submitting!")
    else:
        print(f"\n✅ No issues detected in previewed windows")
    
    # Cost estimate
    avg_tech_tokens = total_tech_tokens / min(max_windows_to_show, len(windows)) if windows else 0
    estimated_total_tech = avg_tech_tokens * len(windows)
    estimated_total_review = 4000 * len(windows)  # Estimate
    
    # Batch API pricing (50% cheaper)
    input_cost = (estimated_total_tech + estimated_total_review) / 1_000_000 * GPT5_MINI_INPUT_PRICE * 0.5
    output_cost = (estimated_total_review * 0.6) / 1_000_000 * GPT5_MINI_OUTPUT_PRICE * 0.5  # Estimate 60% of input
    
    print(f"\n💰 ESTIMATED COST (Batch API - 50% discount):")
    print(f"   Input tokens: ~{(estimated_total_tech + estimated_total_review):,.0f}")
    print(f"   Output tokens: ~{(estimated_total_review * 0.6):,.0f} (estimated)")
    print(f"   Total cost: ~${input_cost + output_cost:.2f}")
    
    print(f"\n{'='*80}")
    print(f"Next steps:")
    print(f"1. If everything looks good: generate_batch_requests_file(...)")
    print(f"2. Review the JSONL file: Open batch_requests.jsonl")
    print(f"3. Submit: submit_batch_job('batch_requests.jsonl')")
    print(f"{'='*80}\n")
    
    return {
        'total_windows': len(windows),
        'flagged_windows': flagged_windows,
        'estimated_cost': input_cost + output_cost
    }


# Quick test
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                 PREVIEW BATCH REQUESTS BEFORE SENDING                        ║
╚══════════════════════════════════════════════════════════════════════════════╝

Use this to check what will be sent BEFORE creating the batch:

>>> preview_batch_requests(
...     start_date="2025-06-01",
...     end_date="2025-11-03",
...     analysis_days=7,
...     review_days=3,
...     stride=3,
...     show_full_prompts=False,  # Set True to see full prompts
...     max_windows_to_show=5     # Number of windows to inspect in detail
... )

This will show you:
✅ What prompts will be sent
✅ Token counts and costs
✅ Potential content policy issues
✅ Data fetch errors

""")


╔══════════════════════════════════════════════════════════════════════════════╗
║                 PREVIEW BATCH REQUESTS BEFORE SENDING                        ║
╚══════════════════════════════════════════════════════════════════════════════╝

Use this to check what will be sent BEFORE creating the batch:

>>> preview_batch_requests(
...     start_date="2025-06-01",
...     end_date="2025-11-03",
...     analysis_days=7,
...     review_days=3,
...     stride=3,
...     show_full_prompts=False,  # Set True to see full prompts
...     max_windows_to_show=5     # Number of windows to inspect in detail
... )

This will show you:
✅ What prompts will be sent
✅ Token counts and costs
✅ Potential content policy issues
✅ Data fetch errors




In [55]:
# ----------------------------------------------------------------------
# Complete OpenAI Batch API Implementation for Backtesting
# ----------------------------------------------------------------------
from openai import OpenAI
import time
from datetime import datetime
import os

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


def create_batch_request(
    custom_id: str,
    system_prompt: str,
    user_prompt: str,
    model: str = "gpt-5-mini"
) -> dict:
    """
    Create a single batch request in OpenAI Batch API format.
    
    Args:
        custom_id: Unique identifier for this request
        system_prompt: System instructions
        user_prompt: User message
        model: Model to use
    
    Returns:
        Dictionary in Batch API format
    """
    return {
        "custom_id": custom_id,
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": model,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        }
    }


def generate_batch_requests_file(
    start_date: str,
    end_date: str,
    analysis_days: int = 7,
    review_days: int = 7,
    stride: int = 3,
    output_file: str = "batch_requests.jsonl",
    model: str = "gpt-5-mini"
) -> dict:
    """
    Generate batch requests file for ALL windows (technical analysis only).
    Review will be done in a second batch after we get technical analysis results.
    
    This creates ONLY technical analysis requests. After you process those results,
    you'll create a second batch for reviews.
    
    Args:
        start_date: Overall start date
        end_date: Overall end date
        analysis_days: Days for analysis period
        review_days: Days for review period
        stride: Days to slide forward
        output_file: Output JSONL file path
        model: Model to use (default: gpt-5-mini)
    
    Returns:
        Dict with metadata about generated requests
    """
    print(f"\n{'='*80}")
    print(f"GENERATING TECHNICAL ANALYSIS BATCH REQUESTS")
    print(f"{'='*80}\n")
    
    # Generate windows
    windows = list(generate_backtest_windows(start_date, end_date, analysis_days, review_days, stride))
    print(f"Total windows: {len(windows)}\n")
    
    # Generate batch requests for technical analysis
    requests = []
    request_metadata = []
    
    for i, (a_start, a_end, r_end) in enumerate(windows):
        # Fetch data for this window
        try:
            data = fetch_backtest_data(a_start, a_end, r_end)
            
            # Technical Analysis Request
            tech_custom_id = f"tech_{a_start}_to_{r_end}"
            tech_user_prompt = f"Phân tích kỹ thuật BTC dựa trên dữ liệu {data['analysis_data']}"
            
            tech_request = create_batch_request(
                custom_id=tech_custom_id,
                system_prompt=technical_analysis_prompt,
                user_prompt=tech_user_prompt,
                model=model
            )
            requests.append(tech_request)
            
            # Store metadata for later processing
            request_metadata.append({
                'custom_id': tech_custom_id,
                'type': 'technical_analysis',
                'analysis_start': a_start,
                'analysis_end': a_end,
                'review_end': r_end,
                'window_index': i
            })
            
            print(f"[{i+1}/{len(windows)}] Generated: {a_start} → {r_end}")
            
        except Exception as e:
            print(f"[{i+1}/{len(windows)}] ERROR: {a_start} → {r_end}: {str(e)}")
    
    # Write to JSONL file
    with open(output_file, 'w', encoding='utf-8') as f:
        for req in requests:
            f.write(json.dumps(req, ensure_ascii=False) + '\n')
    
    # Save metadata
    metadata_file = output_file.replace('.jsonl', '_metadata.json')
    with open(metadata_file, 'w', encoding='utf-8') as f:
        json.dump(request_metadata, f, ensure_ascii=False, indent=2)
    
    print(f"\n{'='*80}")
    print(f"BATCH FILE GENERATED")
    print(f"{'='*80}")
    print(f"Requests file: {output_file}")
    print(f"Metadata file: {metadata_file}")
    print(f"Total requests: {len(requests)}")
    print(f"\n⚠️  IMPORTANT: This batch contains ONLY technical analysis requests.")
    print(f"   After processing these results, you'll need to create a second batch for reviews.")
    print(f"\nNext steps:")
    print(f"  1. Inspect {output_file} to verify all requests")
    print(f"  2. Submit batch: submit_batch_job('{output_file}')")
    print(f"{'='*80}\n")
    
    return {
        'requests_file': output_file,
        'metadata_file': metadata_file,
        'total_requests': len(requests),
        'batch_type': 'technical_analysis'
    }


def submit_batch_job(
    input_file: str,
    description: str = "BTC Backtest Batch"
) -> dict:
    """
    Submit a batch job to OpenAI.
    
    Args:
        input_file: Path to JSONL file with requests
        description: Description for the batch job
    
    Returns:
        Batch job information
    """
    print(f"\n{'='*80}")
    print(f"SUBMITTING BATCH JOB")
    print(f"{'='*80}\n")
    
    # Upload the file
    print(f"📤 Uploading {input_file}...")
    with open(input_file, 'rb') as f:
        batch_input_file = client.files.create(
            file=f,
            purpose="batch"
        )
    
    print(f"✅ File uploaded: {batch_input_file.id}")
    
    # Create batch job
    print(f"🚀 Creating batch job...")
    batch_job = client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={
            "description": description,
            "created_at": datetime.now().isoformat()
        }
    )
    
    print(f"✅ Batch job created!")
    print(f"\n{'='*80}")
    print(f"BATCH JOB DETAILS")
    print(f"{'='*80}")
    print(f"Batch ID: {batch_job.id}")
    print(f"Status: {batch_job.status}")
    print(f"Total requests: {batch_job.request_counts.total}")
    print(f"\nMonitor with: check_batch_status('{batch_job.id}')")
    print(f"{'='*80}\n")
    
    # Save batch info
    batch_info_file = input_file.replace('.jsonl', '_batch_info.json')
    with open(batch_info_file, 'w') as f:
        json.dump({
            'batch_id': batch_job.id,
            'input_file_id': batch_input_file.id,
            'input_file': input_file,
            'status': batch_job.status,
            'created_at': datetime.now().isoformat()
        }, f, indent=2)
    
    print(f"💾 Batch info saved to: {batch_info_file}\n")
    
    return {
        'batch_id': batch_job.id,
        'input_file_id': batch_input_file.id,
        'status': batch_job.status,
        'batch_info_file': batch_info_file
    }


def check_batch_status(batch_id: str) -> dict:
    """
    Check the status of a batch job.
    
    Args:
        batch_id: Batch job ID
    
    Returns:
        Status information
    """
    batch = client.batches.retrieve(batch_id)
    
    print(f"\n{'='*80}")
    print(f"BATCH STATUS")
    print(f"{'='*80}")
    print(f"Batch ID: {batch.id}")
    print(f"Status: {batch.status}")
    print(f"Progress: {batch.request_counts.completed}/{batch.request_counts.total}")
    print(f"  - Completed: {batch.request_counts.completed}")
    print(f"  - Failed: {batch.request_counts.failed}")
    print(f"  - Total: {batch.request_counts.total}")
    
    if batch.status == "completed":
        print(f"\n✅ Batch completed!")
        print(f"Output file ID: {batch.output_file_id}")
        print(f"\nDownload with: download_batch_results('{batch_id}')")
    elif batch.status == "failed":
        print(f"\n❌ Batch failed!")
        if batch.errors:
            print(f"Errors: {batch.errors}")
    elif batch.status in ["in_progress", "validating"]:
        print(f"\n⏳ Batch is still processing...")
    
    print(f"{'='*80}\n")
    
    return {
        'status': batch.status,
        'completed': batch.request_counts.completed,
        'failed': batch.request_counts.failed,
        'total': batch.request_counts.total,
        'output_file_id': batch.output_file_id if batch.status == "completed" else None
    }


def download_batch_results(
    batch_id: str,
    output_file: str = "batch_results.jsonl"
) -> str:
    """
    Download batch results.
    
    Args:
        batch_id: Batch job ID
        output_file: Where to save results
    
    Returns:
        Path to downloaded file
    """
    print(f"\n{'='*80}")
    print(f"DOWNLOADING BATCH RESULTS")
    print(f"{'='*80}\n")
    
    batch = client.batches.retrieve(batch_id)
    
    if batch.status != "completed":
        print(f"❌ Batch not completed yet. Status: {batch.status}")
        return None
    
    print(f"📥 Downloading results from {batch.output_file_id}...")
    
    # Download results
    result_content = client.files.content(batch.output_file_id)
    
    # Save to file
    with open(output_file, 'wb') as f:
        f.write(result_content.content)
    
    print(f"✅ Results saved to: {output_file}")
    
    # Count results
    with open(output_file, 'r', encoding='utf-8') as f:
        results_count = sum(1 for _ in f)
    
    print(f"Total results: {results_count}")
    print(f"\nProcess with: process_batch_results('{output_file}')")
    print(f"{'='*80}\n")
    
    return output_file


def process_batch_results(
    results_file: str,
    metadata_file: str = None,
    output_file: str = "tech_analysis_results.json"
) -> list:
    """
    Process batch results from technical analysis.
    
    Args:
        results_file: JSONL file with batch results
        metadata_file: Metadata file (auto-detected if not provided)
        output_file: Where to save processed results
    
    Returns:
        List of processed results with metadata
    """
    print(f"\n{'='*80}")
    print(f"PROCESSING BATCH RESULTS")
    print(f"{'='*80}\n")
    
    # Load metadata
    if metadata_file is None:
        metadata_file = results_file.replace('_results.jsonl', '_requests_metadata.json').replace('batch_results.jsonl', 'batch_requests_metadata.json')
    
    try:
        with open(metadata_file, 'r', encoding='utf-8') as f:
            metadata = json.load(f)
        metadata_dict = {item['custom_id']: item for item in metadata}
    except FileNotFoundError:
        print(f"⚠️  Metadata file not found: {metadata_file}")
        print(f"   Processing without metadata...")
        metadata_dict = {}
    
    # Load results
    results = []
    with open(results_file, 'r', encoding='utf-8') as f:
        for line in f:
            results.append(json.loads(line))
    
    print(f"Total results: {len(results)}")
    
    # Process each result
    processed_results = []
    success_count = 0
    error_count = 0
    
    for result in results:
        custom_id = result['custom_id']
        meta = metadata_dict.get(custom_id, {})
        
        if result.get('error'):
            print(f"❌ {custom_id}: {result['error']}")
            error_count += 1
            processed_results.append({
                'custom_id': custom_id,
                'status': 'error',
                'error': result['error'],
                'metadata': meta
            })
        else:
            # Extract the response
            response = result['response']['body']['choices'][0]['message']['content']
            usage = result['response']['body']['usage']
            
            processed_results.append({
                'custom_id': custom_id,
                'status': 'success',
                'analysis_start': meta.get('analysis_start'),
                'analysis_end': meta.get('analysis_end'),
                'review_end': meta.get('review_end'),
                'technical_analysis': response,
                'tokens': {
                    'input': usage['prompt_tokens'],
                    'output': usage['completion_tokens'],
                    'total': usage['total_tokens']
                }
            })
            success_count += 1
            print(f"✅ {custom_id}: {usage['total_tokens']} tokens")
    
    # Save processed results
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(processed_results, f, ensure_ascii=False, indent=2)
    
    print(f"\n{'='*80}")
    print(f"PROCESSING COMPLETE")
    print(f"{'='*80}")
    print(f"Success: {success_count}")
    print(f"Errors: {error_count}")
    print(f"Results saved to: {output_file}")
    print(f"\nNext step: Create review batch with:")
    print(f"  generate_review_batch_from_tech_results('{output_file}')")
    print(f"{'='*80}\n")
    
    return processed_results


def generate_review_batch_from_tech_results(
    tech_results_file: str,
    output_file: str = "batch_review_requests.jsonl",
    model: str = "gpt-5-mini"
) -> dict:
    """
    Generate review batch requests from technical analysis results.
    
    Args:
        tech_results_file: JSON file with processed tech analysis results
        output_file: Output JSONL file for review requests
        model: Model to use
    
    Returns:
        Dict with metadata
    """
    print(f"\n{'='*80}")
    print(f"GENERATING REVIEW BATCH REQUESTS")
    print(f"{'='*80}\n")
    
    # Load tech results
    with open(tech_results_file, 'r', encoding='utf-8') as f:
        tech_results = json.load(f)
    
    # Filter successful results
    successful_results = [r for r in tech_results if r['status'] == 'success']
    print(f"Creating review requests for {len(successful_results)} successful analyses\n")
    
    # Generate review requests
    requests = []
    request_metadata = []
    
    for i, result in enumerate(successful_results):
        a_start = result['analysis_start']
        a_end = result['analysis_end']
        r_end = result['review_end']
        tech_analysis = result['technical_analysis']
        
        # Fetch review data
        try:
            data = fetch_backtest_data(a_start, a_end, r_end)
            
            # Create review request
            review_custom_id = f"review_{a_start}_to_{r_end}"
            review_user_prompt = f"""
Đánh giá phân tích kỹ thuật sau đây trong bối cảnh backtesting:

**Thời gian phân tích:** {data['analysis_period']}
**Thời gian đánh giá (thực tế):** {data['review_period']}

**Phân tích kỹ thuật của agent:**
{tech_analysis}

**Dữ liệu thực tế (full period để so sánh):**
{data['full_data']}
"""
            
            review_request = create_batch_request(
                custom_id=review_custom_id,
                system_prompt=reviewer_prompt,
                user_prompt=review_user_prompt,
                model=model
            )
            requests.append(review_request)
            
            # Store metadata
            request_metadata.append({
                'custom_id': review_custom_id,
                'type': 'review',
                'analysis_start': a_start,
                'analysis_end': a_end,
                'review_end': r_end,
                'window_index': i,
                'tech_analysis_tokens': result['tokens']
            })
            
            print(f"[{i+1}/{len(successful_results)}] Generated review: {a_start} → {r_end}")
            
        except Exception as e:
            print(f"[{i+1}/{len(successful_results)}] ERROR: {a_start} → {r_end}: {str(e)}")
    
    # Write to JSONL file
    with open(output_file, 'w', encoding='utf-8') as f:
        for req in requests:
            f.write(json.dumps(req, ensure_ascii=False) + '\n')
    
    # Save metadata
    metadata_file = output_file.replace('.jsonl', '_metadata.json')
    with open(metadata_file, 'w', encoding='utf-8') as f:
        json.dump(request_metadata, f, ensure_ascii=False, indent=2)
    
    print(f"\n{'='*80}")
    print(f"REVIEW BATCH FILE GENERATED")
    print(f"{'='*80}")
    print(f"Requests file: {output_file}")
    print(f"Metadata file: {metadata_file}")
    print(f"Total requests: {len(requests)}")
    print(f"\nNext steps:")
    print(f"  1. Submit batch: submit_batch_job('{output_file}', 'BTC Review Batch')")
    print(f"  2. Monitor: check_batch_status(batch_id)")
    print(f"  3. Download: download_batch_results(batch_id, 'batch_review_results.jsonl')")
    print(f"  4. Process: process_review_results('batch_review_results.jsonl', '{tech_results_file}')")
    print(f"{'='*80}\n")
    
    return {
        'requests_file': output_file,
        'metadata_file': metadata_file,
        'total_requests': len(requests),
        'batch_type': 'review'
    }


def process_review_results(
    review_results_file: str,
    tech_results_file: str,
    metadata_file: str = None,
    output_file: str = "backtest_report.json"
) -> list:
    """
    Process review batch results and combine with technical analysis to create final reports.
    
    Args:
        review_results_file: JSONL file with review batch results
        tech_results_file: JSON file with tech analysis results
        metadata_file: Metadata file (auto-detected if not provided)
        output_file: Where to save final backtest reports
    
    Returns:
        List of complete BacktestReport objects
    """
    print(f"\n{'='*80}")
    print(f"PROCESSING REVIEW RESULTS & CREATING FINAL REPORTS")
    print(f"{'='*80}\n")
    
    # Load tech results
    with open(tech_results_file, 'r', encoding='utf-8') as f:
        tech_results = json.load(f)
    tech_dict = {r['custom_id']: r for r in tech_results}
    
    # Load metadata
    if metadata_file is None:
        metadata_file = review_results_file.replace('_results.jsonl', '_requests_metadata.json').replace('batch_review_results.jsonl', 'batch_review_requests_metadata.json')
    
    try:
        with open(metadata_file, 'r', encoding='utf-8') as f:
            metadata = json.load(f)
        metadata_dict = {item['custom_id']: item for item in metadata}
    except FileNotFoundError:
        print(f"⚠️  Metadata file not found: {metadata_file}")
        metadata_dict = {}
    
    # Load review results
    review_results = []
    with open(review_results_file, 'r', encoding='utf-8') as f:
        for line in f:
            review_results.append(json.loads(line))
    
    print(f"Total review results: {len(review_results)}\n")
    
    # Create final reports
    final_reports = []
    success_count = 0
    error_count = 0
    
    for result in review_results:
        custom_id = result['custom_id']
        meta = metadata_dict.get(custom_id, {})
        
        # Get corresponding tech result
        tech_custom_id = custom_id.replace('review_', 'tech_')
        tech_result = tech_dict.get(tech_custom_id, {})
        
        if result.get('error'):
            print(f"❌ {custom_id}: {result['error']}")
            error_count += 1
            continue
        
        # Extract review response
        review_response = result['response']['body']['choices'][0]['message']['content']
        review_usage = result['response']['body']['usage']
        
        # Create complete report
        report = {
            'analysis_start': meta.get('analysis_start'),
            'analysis_end': meta.get('analysis_end'),
            'review_start': meta.get('analysis_end'),  # Review starts where analysis ends
            'review_end': meta.get('review_end'),
            'analysis': tech_result.get('technical_analysis', ''),
            'review': review_response,
            'analysis_token': tech_result.get('tokens', {}).get('total', 0),
            'review_token': review_usage['total_tokens'],
            'analysis_input_tokens': tech_result.get('tokens', {}).get('input', 0),
            'analysis_output_tokens': tech_result.get('tokens', {}).get('output', 0),
            'review_input_tokens': review_usage['prompt_tokens'],
            'review_output_tokens': review_usage['completion_tokens'],
            'analysis_cost_usd': (tech_result.get('tokens', {}).get('input', 0) * GPT5_MINI_INPUT_PRICE / 1_000_000 * 0.5 +
                                 tech_result.get('tokens', {}).get('output', 0) * GPT5_MINI_OUTPUT_PRICE / 1_000_000 * 0.5),
            'review_cost_usd': (review_usage['prompt_tokens'] * GPT5_MINI_INPUT_PRICE / 1_000_000 * 0.5 +
                               review_usage['completion_tokens'] * GPT5_MINI_OUTPUT_PRICE / 1_000_000 * 0.5),
            'status': 'completed'
        }
        report['total_cost_usd'] = report['analysis_cost_usd'] + report['review_cost_usd']
        
        final_reports.append(report)
        success_count += 1
        print(f"✅ {custom_id}: Complete report created")
    
    # Save final reports
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(final_reports, f, ensure_ascii=False, indent=2)
    
    # Calculate total cost
    total_cost = sum(r['total_cost_usd'] for r in final_reports)
    
    print(f"\n{'='*80}")
    print(f"FINAL REPORTS COMPLETE")
    print(f"{'='*80}")
    print(f"Success: {success_count}")
    print(f"Errors: {error_count}")
    print(f"Total cost: ${total_cost:.2f}")
    print(f"Reports saved to: {output_file}")
    print(f"{'='*80}\n")
    
    return final_reports


# Quick reference
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║              COMPLETE BATCH API WORKFLOW FOR BACKTESTING                     ║
╚══════════════════════════════════════════════════════════════════════════════╝

PHASE 1: Preview (Check what will be sent)
-------------------------------------------
preview = preview_batch_requests(
    start_date="2025-06-01",
    end_date="2025-11-03",
    analysis_days=7,
    review_days=3,
    stride=3,
    max_windows_to_show=5
)

PHASE 2: Generate & Submit Technical Analysis Batch
----------------------------------------------------
# Generate requests file
tech_info = generate_batch_requests_file(
    start_date="2025-06-01",
    end_date="2025-11-03",
    analysis_days=7,
    review_days=3,
    stride=3,
    output_file="batch_tech_requests.jsonl"
)

# Submit to OpenAI
batch1 = submit_batch_job("batch_tech_requests.jsonl", "Tech Analysis Batch")

# Monitor progress
check_batch_status(batch1['batch_id'])

# Download results when complete
download_batch_results(batch1['batch_id'], "batch_tech_results.jsonl")

# Process results
tech_results = process_batch_results(
    "batch_tech_results.jsonl",
    output_file="tech_analysis_results.json"
)

PHASE 3: Generate & Submit Review Batch
----------------------------------------
# Generate review requests from tech results
review_info = generate_review_batch_from_tech_results(
    tech_results_file="tech_analysis_results.json",
    output_file="batch_review_requests.jsonl"
)

# Submit review batch
batch2 = submit_batch_job("batch_review_requests.jsonl", "Review Batch")

# Monitor progress
check_batch_status(batch2['batch_id'])

# Download results when complete
download_batch_results(batch2['batch_id'], "batch_review_results.jsonl")

# Process and create final reports
final_reports = process_review_results(
    review_results_file="batch_review_results.jsonl",
    tech_results_file="tech_analysis_results.json",
    output_file="backtest_report.json"
)

BENEFITS:
✅ 50% CHEAPER than real-time API
✅ Preview all requests before sending
✅ Two-phase processing (tech → review)
✅ No rate limits
✅ Inspect what triggered policy errors

""")


╔══════════════════════════════════════════════════════════════════════════════╗
║              COMPLETE BATCH API WORKFLOW FOR BACKTESTING                     ║
╚══════════════════════════════════════════════════════════════════════════════╝

PHASE 1: Preview (Check what will be sent)
-------------------------------------------
preview = preview_batch_requests(
    start_date="2025-06-01",
    end_date="2025-11-03",
    analysis_days=7,
    review_days=3,
    stride=3,
    max_windows_to_show=5
)

PHASE 2: Generate & Submit Technical Analysis Batch
----------------------------------------------------
# Generate requests file
tech_info = generate_batch_requests_file(
    start_date="2025-06-01",
    end_date="2025-11-03",
    analysis_days=7,
    review_days=3,
    stride=3,
    output_file="batch_tech_requests.jsonl"
)

# Submit to OpenAI
batch1 = submit_batch_job("batch_tech_requests.jsonl", "Tech Analysis Batch")

# Monitor progress
check_batch_status(batch1['batch_id'])

# Downloa

In [ ]:
starting_date = "2025-05-13"
ending_date = "2025-10-31"
visualize_windows(starting_date, ending_date, 7, 3, stride=3)

reports = await execute_full_backtest(
    start_date=starting_date,
    end_date=ending_date,
    analysis_days=7,
    review_days=3,
    stride=3,           # 3-day sliding window
    parallel=True,
    output_file="backtest_report.json"
)






BACKTEST WINDOWS STRUCTURE (SLIDING WINDOW)
Period: 2025-05-13 → 2025-10-31
Window size: 7 days analysis + 7 days review = 14 days total
Stride: 3 days (sliding window - overlapping)
Total windows: 53

Window  1:
   Analysis: 2025-05-13 → 2025-05-20 (agent makes predictions)
   Review:   2025-05-20 → 2025-05-27 (compare vs actual)

Window  2:
   Analysis: 2025-05-16 → 2025-05-23 (agent makes predictions)
   Review:   2025-05-23 → 2025-05-30 (compare vs actual)

Window  3:
   Analysis: 2025-05-19 → 2025-05-26 (agent makes predictions)
   Review:   2025-05-26 → 2025-06-02 (compare vs actual)

Window  4:
   Analysis: 2025-05-22 → 2025-05-29 (agent makes predictions)
   Review:   2025-05-29 → 2025-06-05 (compare vs actual)

Window  5:
   Analysis: 2025-05-25 → 2025-06-01 (agent makes predictions)
   Review:   2025-06-01 → 2025-06-08 (compare vs actual)

Window  6:
   Analysis: 2025-05-28 → 2025-06-04 (agent makes predictions)
   Review:   2025-06-04 → 2025-06-11 (compare vs actual)

Windo

Error getting response: Error code: 400 - {'error': {'message': 'Invalid prompt: your prompt was flagged as potentially violating our usage policy. Please try again with a different prompt: https://platform.openai.com/docs/guides/reasoning#advice-on-prompting', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_prompt'}}. (request_id: req_a8ad3913edc945b48aea6326fd3b1823)


BadRequestError: Error code: 400 - {'error': {'message': 'Invalid prompt: your prompt was flagged as potentially violating our usage policy. Please try again with a different prompt: https://platform.openai.com/docs/guides/reasoning#advice-on-prompting', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_prompt'}}

✓ Review completed (2025-07-25 → 2025-07-28)
  Tokens: 6,322 in + 2,021 out = 8,343
  Cost: $0.005622

────────────────────────────────────────────────────────────
Window Total: $0.010403
────────────────────────────────────────────────────────────

✓ Review completed (2025-06-22 → 2025-06-25)
  Tokens: 6,143 in + 2,298 out = 8,441
  Cost: $0.006132

────────────────────────────────────────────────────────────
Window Total: $0.010553
────────────────────────────────────────────────────────────

✓ Review completed (2025-06-16 → 2025-06-19)
  Tokens: 6,140 in + 2,314 out = 8,454
  Cost: $0.006163

────────────────────────────────────────────────────────────
Window Total: $0.010575
────────────────────────────────────────────────────────────

✓ Review completed (2025-05-26 → 2025-05-29)
  Tokens: 5,992 in + 2,209 out = 8,201
  Cost: $0.005916

────────────────────────────────────────────────────────────
Window Total: $0.010025
────────────────────────────────────────────────────────────



In [49]:
preview = preview_batch_requests(
      start_date="2025-05-13",
      end_date="2025-10-31",
      analysis_days=7,
      review_days=3,
      stride=3,
      show_full_prompts=True,  # Set True to see actual prompts
      max_windows_to_show=5     # Check first 5 windows in detail
  )


BATCH REQUESTS PREVIEW - What will be sent to OpenAI

Total windows to process: 54

DETAILED ANALYSIS (first 5 windows)


────────────────────────────────────────────────────────────────────────────────
Window 1: 2025-05-13 → 2025-05-20 → 2025-05-23
────────────────────────────────────────────────────────────────────────────────

📊 TECHNICAL ANALYSIS REQUEST:
   System prompt: 1,526 chars
   User prompt: 6,883 chars
   Total tokens: ~3,052

📝 REVIEW REQUEST (will be created after tech analysis):
   System prompt: 2,056 chars
   User prompt: Will include tech analysis + review data
   Estimated tokens: ~3000-5000

✅ No obvious issues detected

────────────────────────────────────────
FULL TECHNICAL ANALYSIS PROMPT:
────────────────────────────────────────
System: 
Bạn là chuyên gia phân tích kỹ thuật crypto chuyên sâu, sử dụng logic phân tích đa khung thời gian 4H kết hợp dữ liệu thực tế gồm:
- Khung giờ mà bạn sẽ được cung cấp cũng như sử dụng là giờ Việt Nam...
User: Phân tích kỹ thu

In [50]:
  # STEP 2: Generate technical analysis batch
tech_info = generate_batch_requests_file(
      start_date="2025-06-01",
      end_date="2025-11-03",
      analysis_days=7,
      review_days=3,
      stride=3,
      output_file="batch_tech_requests.jsonl"
  )


GENERATING TECHNICAL ANALYSIS BATCH REQUESTS

Total windows: 49

[1/49] Generated: 2025-06-01 → 2025-06-11
[2/49] Generated: 2025-06-04 → 2025-06-14
[3/49] Generated: 2025-06-07 → 2025-06-17
[4/49] Generated: 2025-06-10 → 2025-06-20
[5/49] Generated: 2025-06-13 → 2025-06-23
[6/49] Generated: 2025-06-16 → 2025-06-26
[7/49] Generated: 2025-06-19 → 2025-06-29
[8/49] Generated: 2025-06-22 → 2025-07-02
[9/49] Generated: 2025-06-25 → 2025-07-05
[10/49] Generated: 2025-06-28 → 2025-07-08
[11/49] Generated: 2025-07-01 → 2025-07-11
[12/49] Generated: 2025-07-04 → 2025-07-14
[13/49] Generated: 2025-07-07 → 2025-07-17
[14/49] Generated: 2025-07-10 → 2025-07-20
[15/49] Generated: 2025-07-13 → 2025-07-23
[16/49] Generated: 2025-07-16 → 2025-07-26
[17/49] Generated: 2025-07-19 → 2025-07-29
[18/49] Generated: 2025-07-22 → 2025-08-01
[19/49] Generated: 2025-07-25 → 2025-08-04
[20/49] Generated: 2025-07-28 → 2025-08-07
[21/49] Generated: 2025-07-31 → 2025-08-10
[22/49] Generated: 2025-08-03 → 2025-08-

In [ ]:
batch1 = submit_batch_job("batch_tech_requests.jsonl", "Tech Analysis")
check_batch_status(batch1['batch_id'])  # Run periodically

In [56]:
review_info = generate_review_batch_from_tech_results("tech_analysis_results.json")


GENERATING REVIEW BATCH REQUESTS

Creating review requests for 49 successful analyses

[1/49] Generated review: 2025-06-01 → 2025-06-11
[2/49] Generated review: 2025-06-04 → 2025-06-14
[3/49] Generated review: 2025-06-07 → 2025-06-17
[4/49] Generated review: 2025-06-10 → 2025-06-20
[5/49] Generated review: 2025-06-13 → 2025-06-23
[6/49] Generated review: 2025-06-16 → 2025-06-26
[7/49] Generated review: 2025-06-19 → 2025-06-29
[8/49] Generated review: 2025-06-22 → 2025-07-02
[9/49] Generated review: 2025-06-25 → 2025-07-05
[10/49] Generated review: 2025-06-28 → 2025-07-08
[11/49] Generated review: 2025-07-01 → 2025-07-11
[12/49] Generated review: 2025-07-04 → 2025-07-14
[13/49] Generated review: 2025-07-07 → 2025-07-17
[14/49] Generated review: 2025-07-10 → 2025-07-20
[15/49] Generated review: 2025-07-13 → 2025-07-23
[16/49] Generated review: 2025-07-16 → 2025-07-26
[17/49] Generated review: 2025-07-19 → 2025-07-29
[18/49] Generated review: 2025-07-22 → 2025-08-01
[19/49] Generated rev

In [ ]:
  # STEP 6: Submit & process reviews
batch2 = submit_batch_job("batch_review_requests.jsonl", "Reviews")
  # ... monitor, download, process as before




SUBMITTING BATCH JOB

📤 Uploading batch_review_requests.jsonl...
✅ File uploaded: file-CX7n6xGnoYmcwjH9xnnJPp
🚀 Creating batch job...
✅ Batch job created!

BATCH JOB DETAILS
Batch ID: batch_6909c2956ac481908318966a8528c28d
Status: validating
Total requests: 0

Monitor with: check_batch_status('batch_6909c2956ac481908318966a8528c28d')

💾 Batch info saved to: batch_review_requests_batch_info.json


PROCESSING REVIEW RESULTS & CREATING FINAL REPORTS



FileNotFoundError: [Errno 2] No such file or directory: 'batch_review_results.jsonl'

In [59]:
final_reports = process_review_results("batch_review_results.jsonl", "tech_analysis_results.json")


PROCESSING REVIEW RESULTS & CREATING FINAL REPORTS

Total review results: 49

✅ review_2025-06-01_to_2025-06-11: Complete report created
✅ review_2025-06-04_to_2025-06-14: Complete report created
✅ review_2025-06-07_to_2025-06-17: Complete report created
✅ review_2025-06-10_to_2025-06-20: Complete report created
✅ review_2025-06-13_to_2025-06-23: Complete report created
✅ review_2025-06-16_to_2025-06-26: Complete report created
✅ review_2025-06-19_to_2025-06-29: Complete report created
✅ review_2025-06-22_to_2025-07-02: Complete report created
✅ review_2025-06-25_to_2025-07-05: Complete report created
✅ review_2025-06-28_to_2025-07-08: Complete report created
✅ review_2025-07-01_to_2025-07-11: Complete report created
✅ review_2025-07-04_to_2025-07-14: Complete report created
✅ review_2025-07-07_to_2025-07-17: Complete report created
✅ review_2025-07-10_to_2025-07-20: Complete report created
✅ review_2025-07-13_to_2025-07-23: Complete report created
✅ review_2025-07-16_to_2025-07-26: C

In [60]:
import json
import csv

# Đường dẫn file
input_file = "backtest_report.json"
output_file = "backtest_report.csv"

# Đọc file JSON
with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

# Chỉ lấy các trường cần thiết
fields = ["analysis_start", "analysis_end", "review_start", "review_end", "analysis", "review"]

# Ghi ra file CSV
with open(output_file, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for item in data:
        filtered_item = {key: item.get(key, "") for key in fields}
        writer.writerow(filtered_item)

print(f"✅ Đã xuất file CSV thành công: {output_file}")


✅ Đã xuất file CSV thành công: backtest_report.csv
